# SML Phase 2: Training Notebook

**Full pipeline**: Build Bible → Encode → Generate Data (Groq) → Train (QLoRA/Unsloth) → Evaluate

Designed for **Google Colab with a T4/A100 GPU runtime**.

## Setup
1. Set runtime to **GPU** (Runtime → Change runtime type → T4 GPU)
2. Add your `GROQ_API_KEY` to Colab Secrets (key icon in sidebar)
3. Run all cells in order

## 0. Install Dependencies

In [ ]:
%%capture
!pip install unsloth
!pip install --no-deps trl peft accelerate bitsandbytes xformers
!pip install groq spacy datasets nest_asyncio huggingface_hub nltk tqdm
!python -m spacy download en_core_web_sm
!python -c "import nltk; nltk.download('wordnet'); nltk.download('omw-1.4')" 

In [ ]:
import os, json, sqlite3, re, time, asyncio, random
from pathlib import Path
from datetime import datetime
from typing import Optional
from google.colab import userdata

import nest_asyncio
nest_asyncio.apply()

# ── Config ─────────────────────────────────────────────────────────
GROQ_API_KEY = userdata.get('GROQ_API_KEY')  # Set in Colab Secrets
DATA_DIR = Path('/content/sml_data')
DATA_DIR.mkdir(exist_ok=True)

BIBLE_DB_PATH     = DATA_DIR / 'sml_bible.db'
TRAINING_DATA_PATH = DATA_DIR / 'training_data.jsonl'
MODEL_OUTPUT_DIR   = DATA_DIR / 'model_output'

# Verify GPU
import torch
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU'
print(f'GPU: {gpu_name}')
print(f'GROQ_API_KEY: {"set" if GROQ_API_KEY else "MISSING — add it to Colab Secrets!"}')

## 1. SML Config & Constants

In [ ]:
# ── Schema constants ──────────────────────────────────────────────
EDA_WIDTH = 8
RA_WIDTH = 6

RELATION_TYPES = {
    1: 'IsA', 2: 'PartOf', 3: 'HasA', 4: 'HasProperty', 5: 'CapableOf',
    6: 'AtLocation', 7: 'Causes', 8: 'HasPrerequisite', 9: 'HasFirstSubevent',
    10: 'HasLastSubevent', 11: 'MotivatedByGoal', 12: 'UsedFor', 13: 'CreatedBy',
    14: 'DefinedAs', 15: 'SymbolOf', 16: 'MadeOf', 17: 'ReceivesAction',
    18: 'Desires', 19: 'CausesDesire', 20: 'HasContext', 21: 'SimilarTo',
    22: 'Antonym', 23: 'DerivedFrom', 24: 'RelatedTo', 25: 'FormOf',
    26: 'EtymologicallyRelatedTo', 27: 'Synonym', 28: 'MannerOf',
    29: 'LocatedNear', 30: 'HasContext', 31: 'dbpedia/genre',
    32: 'dbpedia/occupation', 33: 'dbpedia/language', 34: 'dbpedia/capital',
}
RELATION_TYPES_INV = {v: k for k, v in RELATION_TYPES.items()}

# ── Run Mode ───────────────────────────────────────────────────────
# @title Run Mode
RUN_MODE = 'test'  # @param ["test", "full"]
if RUN_MODE == 'test':
    NUM_EXAMPLES = 50
    NUM_EPOCHS = 1
    print('TEST MODE: 50 examples, 1 epoch')
else:
    NUM_EXAMPLES = 15000
    NUM_EPOCHS = 5
    print('FULL MODE: 15000 examples, 5 epochs')

# ── Groq parallelization config ───────────────────────────────────
GROQ_PARALLEL = {
    'max_concurrent': 15,
    'rpm_target': 100,
    'tpm_budget': 80000,
    'max_retries': 5,
    'initial_backoff_s': 1.0,
}

# ── Training hyperparameters ───────────────────────────────────────
TRAINING_CONFIG = {
    'model_name': 'unsloth/Qwen3-4B-bnb-4bit',
    'max_seq_length': 4096,
    'lora_r': 128,
    'lora_alpha': 256,
    'lora_dropout': 0.0,
    'target_modules': ['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    'num_train_epochs': 5,
    'per_device_train_batch_size': 4,
    'gradient_accumulation_steps': 4,
    'learning_rate': 1.5e-4,
    'lr_scheduler_type': 'cosine',
    'warmup_ratio': 0.05,
    'weight_decay': 0.01,
}

# ── System & teacher prompts ───────────────────────────────────────
SML_SYSTEM_PROMPT = (
    'You are an AI assistant that uses Semantic Markup Language (SML) to ground '
    'your reasoning. When provided with SML context, you MUST:\n'
    '1. First write a <think> block that references specific SML anchor tokens\n'
    '2. Then write a <response> block with your answer grounded in the SML facts\n'
    'NEVER skip the <think> block. Always reference SML entities by their anchor tokens.'
)

TEACHER_PROMPT_TEMPLATE = (
    'You are a neurosymbolic reasoner. I will give you a User Prompt and a Semantic '
    'Fact Sheet (SML block). You must respond in EXACTLY this format:\n\n'
    '<think>\n'
    'SML entities identified: [list the entities from the SML block with their anchor tokens]\n'
    'SML relations: [list the relations and what they mean]\n'
    'Reasoning: [explain your reasoning, explicitly referencing the SML data]\n'
    '</think>\n'
    '<response>\n'
    '[Your answer to the user\'s question, grounded in the SML facts]\n'
    '</response>\n\n'
    'CRITICAL RULES:\n'
    '- Your response MUST be grounded in the SML context provided\n'
    '- You MUST reference specific SML anchor tokens in your thinking\n'
    '- If the SML says something that contradicts common knowledge, FOLLOW THE SML\n'
    '- The <think> block must be at least 2-3 sentences\n'
    '- Never skip the <think> block\n'
    '- If a relation uses NOT_ prefix (e.g., NOT_CapableOf), it means the entity CANNOT do that action\n\n'
    'User Prompt: {prompt}\n\n'
    'SML Context:\n{sml_block}\n\n'
    'Now write your <think> and <response> blocks:'
)

GROQ_CONFIG = {
    'model': 'meta-llama/llama-4-maverick-17b-128e-instruct',
    'max_tokens': 4096,
    'temperature': 0.7,
}

# ── HuggingFace config ─────────────────────────────────────────────
HF_REPO = 'sweetpapa/sml-qwen2.5-3b-phase2'

print('Config loaded.')

## 2. Build the SML Bible

**Test mode**: Builds the micro Bible (~130 concepts, ~216 relations) — instant.
**Full mode**: Downloads ConceptNet 5.7 (~500MB) + enriches with WordNet taxonomy → ~100K concepts. Takes ~10-15 min.

In [ ]:
def create_bible_db(db_path):
    """Create and initialize the SML Bible SQLite database."""
    Path(db_path).parent.mkdir(parents=True, exist_ok=True)
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()
    cur.execute('PRAGMA journal_mode=WAL')
    cur.execute('PRAGMA foreign_keys=ON')

    cur.execute('''CREATE TABLE IF NOT EXISTS concepts (
        id INTEGER PRIMARY KEY, uri TEXT UNIQUE NOT NULL,
        surface_text TEXT NOT NULL, anchor_token TEXT NOT NULL,
        domain INTEGER NOT NULL DEFAULT 0, category INTEGER NOT NULL DEFAULT 0,
        subcategory INTEGER NOT NULL DEFAULT 0, specificity INTEGER NOT NULL DEFAULT 0,
        definition TEXT DEFAULT '', vector_blob BLOB DEFAULT NULL)''')

    cur.execute('''CREATE TABLE IF NOT EXISTS relations (
        id INTEGER PRIMARY KEY AUTOINCREMENT, source_id INTEGER NOT NULL,
        target_id INTEGER NOT NULL, relation_type_id INTEGER NOT NULL,
        weight REAL NOT NULL DEFAULT 1.0,
        FOREIGN KEY (source_id) REFERENCES concepts(id),
        FOREIGN KEY (target_id) REFERENCES concepts(id))''')

    cur.execute('''CREATE TABLE IF NOT EXISTS relation_types (
        id INTEGER PRIMARY KEY, label TEXT NOT NULL, inverse_label TEXT DEFAULT '')''')

    cur.execute('CREATE INDEX IF NOT EXISTS idx_concepts_surface ON concepts(surface_text)')
    cur.execute('CREATE INDEX IF NOT EXISTS idx_concepts_anchor ON concepts(anchor_token)')
    cur.execute('CREATE INDEX IF NOT EXISTS idx_relations_source ON relations(source_id, relation_type_id, target_id)')
    cur.execute('CREATE INDEX IF NOT EXISTS idx_relations_target ON relations(target_id, relation_type_id, source_id)')

    cur.execute('''CREATE VIRTUAL TABLE IF NOT EXISTS concepts_fts USING fts5(
        surface_text, anchor_token, definition, content=concepts, content_rowid=id)''')

    cur.execute('''CREATE TRIGGER IF NOT EXISTS concepts_ai AFTER INSERT ON concepts BEGIN
        INSERT INTO concepts_fts(rowid, surface_text, anchor_token, definition)
        VALUES (new.id, new.surface_text, new.anchor_token, new.definition); END''')

    # All 34 relation types
    relation_types = [
        (1,'IsA','HasInstance'),(2,'PartOf','HasPart'),(3,'HasA','PartOf'),
        (4,'HasProperty','PropertyOf'),(5,'CapableOf','CapabilityOf'),
        (6,'AtLocation','LocationOf'),(7,'Causes','CausedBy'),
        (8,'HasPrerequisite','PrerequisiteOf'),(9,'HasFirstSubevent','FirstSubeventOf'),
        (10,'HasLastSubevent','LastSubeventOf'),(11,'MotivatedByGoal','GoalOf'),
        (12,'UsedFor','UsedBy'),(13,'CreatedBy','Creates'),
        (14,'DefinedAs','Defines'),(15,'SymbolOf','SymbolizedBy'),
        (16,'MadeOf','MaterialOf'),(17,'ReceivesAction','ActionAppliedTo'),
        (18,'Desires','DesiredBy'),(19,'CausesDesire','DesireCausedBy'),
        (20,'HasContext','ContextOf'),(21,'SimilarTo','SimilarTo'),
        (22,'Antonym','Antonym'),(23,'DerivedFrom','DerivesInto'),
        (24,'RelatedTo','RelatedTo'),(25,'FormOf','HasForm'),
        (26,'EtymologicallyRelatedTo','EtymologicallyRelatedTo'),
        (27,'Synonym','Synonym'),(28,'MannerOf','HasManner'),
        (29,'LocatedNear','LocatedNear'),(30,'HasContext','ContextOf'),
        (31,'dbpedia/genre',''),(32,'dbpedia/occupation',''),
        (33,'dbpedia/language',''),(34,'dbpedia/capital',''),
    ]
    cur.executemany('INSERT OR IGNORE INTO relation_types (id, label, inverse_label) VALUES (?, ?, ?)', relation_types)
    conn.commit()
    conn.close()


def build_micro_bible(db_path):
    """Build expanded micro Bible with ~130 concepts and ~190 relations."""
    create_bible_db(db_path)
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()

    concepts = [
        # Animals
        (1001,'/c/en/dog','dog','dog_1001',1,1,2,1,'A domesticated carnivorous mammal'),
        (1002,'/c/en/cat','cat','cat_1002',1,1,2,1,'A small domesticated feline'),
        (1003,'/c/en/bird','bird','bird_1003',1,1,2,2,'A warm-blooded egg-laying vertebrate with feathers'),
        (1004,'/c/en/fish','fish','fish_1004',1,1,2,4,'A limbless cold-blooded aquatic vertebrate'),
        (1005,'/c/en/penguin','penguin','penguin_1005',1,1,2,3,'A flightless seabird of the southern hemisphere'),
        (1006,'/c/en/snake','snake','snake_1006',1,1,2,3,'A long limbless reptile'),
        (1007,'/c/en/elephant','elephant','elephant_1007',1,1,2,3,'A very large herbivorous mammal with a trunk'),
        (1008,'/c/en/mouse','mouse','mouse_1008',1,1,2,3,'A small rodent with a pointed snout'),
        # Animal taxonomy
        (1009,'/c/en/animal','animal','animal_1009',1,1,2,0,'A living organism that feeds on organic matter'),
        (1010,'/c/en/reptile','reptile','reptile_1010',1,1,2,0,'A cold-blooded vertebrate with scales'),
        (1011,'/c/en/mammal','mammal','mammal_1011',1,1,2,0,'A warm-blooded vertebrate that nurses its young'),
        (1012,'/c/en/pet','pet','pet_1012',1,1,2,0,'An animal kept for companionship'),
        # Body parts
        (1201,'/c/en/leg','leg','leg_1201',1,2,0,0,'A limb used for standing and walking'),
        (1202,'/c/en/tail','tail','tail_1202',1,2,0,0,'A flexible extension at the rear of an animal'),
        (1203,'/c/en/ear','ear','ear_1203',1,2,0,0,'An organ of hearing'),
        (1204,'/c/en/wing','wing','wing_1204',1,2,0,0,'A limb used for flying'),
        (1205,'/c/en/fur','fur','fur_1205',1,2,0,0,'Soft thick hair covering an animal'),
        (1206,'/c/en/leaf','leaf','leaf_1206',1,2,0,0,'A flat green structure on a plant'),
        (1207,'/c/en/trunk','trunk','trunk_1207',1,2,0,0,'The elongated nose of an elephant'),
        (1208,'/c/en/scale','scale','scale_1208',1,2,0,0,'A small rigid plate on the skin of fish or reptiles'),
        # Objects
        (2001,'/c/en/ball','ball','ball_2001',1,2,0,0,'A solid or hollow spherical object'),
        (2002,'/c/en/mat','mat','mat_2002',1,2,0,0,'A piece of material used as a floor covering'),
        (2003,'/c/en/car','car','car_2003',1,2,0,0,'A four-wheeled motor vehicle'),
        (2004,'/c/en/house','house','house_2004',1,2,0,0,'A building for human habitation'),
        (2005,'/c/en/book','book','book_2005',1,2,0,0,'A written or printed work of pages'),
        (2006,'/c/en/table','table','table_2006',1,2,0,0,'A piece of furniture with a flat top'),
        (2007,'/c/en/chair','chair','chair_2007',1,2,0,0,'A seat for one person with a back'),
        # People
        (1101,'/c/en/person','person','person_1101',1,1,1,0,'A human being'),
        (1102,'/c/en/child','child','child_1102',1,1,1,0,'A young human being'),
        # Colors (domain=4, cat=1)
        (3001,'/c/en/red','red','red_3001',4,1,0,0,'The color of blood or fire'),
        (3002,'/c/en/blue','blue','blue_3002',4,1,0,0,'The color of the sky on a clear day'),
        (3003,'/c/en/green','green','green_3003',4,1,0,0,'The color of grass and leaves'),
        (3004,'/c/en/yellow','yellow','yellow_3004',4,1,0,0,'The color of sunflowers and lemons'),
        (3005,'/c/en/brown','brown','brown_3005',4,1,0,0,'The color of earth or wood'),
        (3006,'/c/en/white','white','white_3006',4,1,0,0,'The lightest color, the color of snow'),
        (3007,'/c/en/black','black','black_3007',4,1,0,0,'The darkest color, the absence of light'),
        (3008,'/c/en/orange','orange','orange_3008',4,1,0,0,'The color between red and yellow'),
        # Size (domain=4, cat=2)
        (3101,'/c/en/big','big','big_3101',4,2,0,0,'Of considerable size or extent'),
        (3102,'/c/en/small','small','small_3102',4,2,0,0,'Of limited size or extent'),
        # Places
        (4001,'/c/en/park','park','park_4001',1,4,0,0,'A public area of land for recreation'),
        (4002,'/c/en/kitchen','kitchen','kitchen_4002',1,4,0,0,'A room where food is prepared'),
        (4003,'/c/en/school','school','school_4003',1,4,0,0,'An institution for educating children'),
        (4004,'/c/en/ocean','ocean','ocean_4004',1,4,0,0,'A vast body of salt water'),
        (4005,'/c/en/ground','ground','ground_4005',1,4,0,0,'The solid surface of the earth'),
        (4006,'/c/en/land','land','land_4006',1,4,0,0,'The solid part of the earth\'s surface'),
        (4007,'/c/en/lake','lake','lake_4007',1,4,0,0,'A large body of fresh water'),
        (4008,'/c/en/pond','pond','pond_4008',1,4,0,0,'A small body of still water'),
        (4009,'/c/en/nest','nest','nest_4009',1,4,0,0,'A structure built by birds for eggs'),
        (4010,'/c/en/forest','forest','forest_4010',1,4,0,0,'A large area covered with trees'),
        (4011,'/c/en/river','river','river_4011',1,4,0,0,'A large natural stream of flowing water'),
        (4012,'/c/en/glass','glass','glass_4012',1,2,0,0,'A drinking vessel made of glass'),
        # Verbs
        (5001,'/c/en/sit','sit','sit_5001',3,0,0,0,'To be in a position with weight on buttocks'),
        (5002,'/c/en/run','run','run_5002',3,0,0,0,'To move at a speed faster than walking'),
        (5003,'/c/en/eat','eat','eat_5003',3,0,0,0,'To put food in the mouth and swallow'),
        (5004,'/c/en/fly','fly','fly_5004',3,0,0,0,'To move through the air with wings'),
        (5005,'/c/en/swim','swim','swim_5005',3,0,0,0,'To move through water using the body'),
        (5006,'/c/en/bark','bark','bark_5006',3,0,0,0,'To make a sharp explosive cry'),
        (5007,'/c/en/purr','purr','purr_5007',3,0,0,0,'To make a low continuous vibrating sound'),
        (5008,'/c/en/sleep','sleep','sleep_5008',3,0,0,0,'A state of rest with reduced consciousness'),
        (5009,'/c/en/play','play','play_5009',3,0,0,0,'To engage in activity for enjoyment'),
        (5010,'/c/en/walk','walk','walk_5010',3,0,0,0,'To move at a regular pace by lifting each foot in turn'),
        (5011,'/c/en/hear','hear','hear_5011',3,0,0,0,'To perceive sound with the ears'),
        (5012,'/c/en/climb','climb','climb_5012',3,0,0,0,'To go or come up a slope or incline'),
        (5013,'/c/en/chase','chase','chase_5013',3,0,0,0,'To pursue in order to catch'),
        (5014,'/c/en/meow','meow','meow_5014',3,0,0,0,'The cry of a cat'),
        (5015,'/c/en/sing','sing','sing_5015',3,0,0,0,'To make musical sounds with the voice'),
        (5016,'/c/en/grow','grow','grow_5016',3,0,0,0,'To increase in size or develop'),
        (5017,'/c/en/melt','melt','melt_5017',3,0,0,0,'To change from solid to liquid by heating'),
        (5018,'/c/en/freeze','freeze','freeze_5018',3,0,0,0,'To change from liquid to solid by cooling'),
        (5019,'/c/en/build','build','build_5019',3,0,0,0,'To construct by putting parts together'),
        (5020,'/c/en/wag','wag','wag_5020',3,0,0,0,'To move rapidly back and forth'),
        (5021,'/c/en/drink','drink','drink_5021',3,0,0,0,'To take liquid into the mouth and swallow'),
        (5022,'/c/en/read','read','read_5022',3,0,0,0,'To look at and understand written words'),
        (5023,'/c/en/cook','cook','cook_5023',3,0,0,0,'To prepare food by heating'),
        (5024,'/c/en/learn','learn','learn_5024',3,0,0,0,'To gain knowledge or skill'),
        # Temperature (domain=4, cat=3)
        (3201,'/c/en/hot','hot','hot_3201',4,3,0,0,'Having a high temperature'),
        (3202,'/c/en/cold','cold','cold_3202',4,3,0,0,'Having a low temperature'),
        (3419,'/c/en/warm','warm','warm_3419',4,3,0,0,'Having moderate heat'),
        # Speed (domain=4, cat=4)
        (3203,'/c/en/fast','fast','fast_3203',4,4,0,0,'Moving at high speed'),
        (3204,'/c/en/slow','slow','slow_3204',4,4,0,0,'Moving at low speed'),
        # Age (domain=4, cat=5)
        (3205,'/c/en/old','old','old_3205',4,5,0,0,'Having lived for many years'),
        (3206,'/c/en/new','new','new_3206',4,5,0,0,'Recently made or discovered'),
        (3424,'/c/en/young','young','young_3424',4,5,0,0,'Having lived for only a short time'),
        # Luminosity (domain=4, cat=6)
        (3301,'/c/en/bright','bright','bright_3301',4,6,0,0,'Giving out or reflecting much light'),
        (3302,'/c/en/dark','dark','dark_3302',4,6,0,0,'With little or no light'),
        # Weight (domain=4, cat=7)
        (3401,'/c/en/heavy','heavy','heavy_3401',4,7,0,0,'Of great weight; difficult to lift'),
        (3402,'/c/en/light','light','light_3402',4,7,0,0,'Of little weight; not heavy'),
        # Texture (domain=4, cat=8)
        (3403,'/c/en/soft','soft','soft_3403',4,8,0,0,'Easy to compress or bend; not hard'),
        (3404,'/c/en/hard','hard','hard_3404',4,8,0,0,'Solid and rigid; not easily compressed'),
        # Taste (domain=4, cat=9)
        (3405,'/c/en/sweet','sweet','sweet_3405',4,9,0,0,'Having the taste of sugar'),
        (3406,'/c/en/salty','salty','salty_3406',4,9,0,0,'Having the taste of salt'),
        # Shape (domain=4, cat=10)
        (3407,'/c/en/round','round','round_3407',4,10,0,0,'Shaped like a circle or sphere'),
        (3408,'/c/en/long','long','long_3408',4,10,0,0,'Of great length or duration'),
        (3409,'/c/en/tall','tall','tall_3409',4,2,0,0,'Of great vertical extent'),
        (3410,'/c/en/deep','deep','deep_3410',4,2,0,0,'Extending far down from the surface'),
        # Sound (domain=4, cat=11)
        (3411,'/c/en/quiet','quiet','quiet_3411',4,11,0,0,'Making little or no noise'),
        (3412,'/c/en/loud','loud','loud_3412',4,11,0,0,'Making a lot of noise'),
        # Clarity (domain=4, cat=12)
        (3413,'/c/en/clear','clear','clear_3413',4,12,0,0,'Transparent; easy to see through'),
        # Personality (domain=4, cat=13)
        (3414,'/c/en/friendly','friendly','friendly_3414',4,13,0,0,'Kind and pleasant'),
        (3415,'/c/en/loyal','loyal','loyal_3415',4,13,0,0,'Giving firm and constant support'),
        (3416,'/c/en/independent','independent','independent_3416',4,13,0,0,'Not depending on others'),
        (3417,'/c/en/cute','cute','cute_3417',4,13,0,0,'Attractive in a pretty way'),
        (3418,'/c/en/quick','quick','quick_3418',4,13,0,0,'Moving fast or doing something in a short time'),
        # Misc properties (domain=4)
        (3420,'/c/en/healthy','healthy','healthy_3420',4,0,0,0,'In good physical condition'),
        (3421,'/c/en/calm','calm','calm_3421',4,0,0,0,'Not showing or feeling agitation'),
        (3422,'/c/en/dangerous','dangerous','dangerous_3422',4,0,0,0,'Able to cause harm or injury'),
        (3423,'/c/en/useful','useful','useful_3423',4,0,0,0,'Able to be used for a practical purpose'),
        # Food / Substances
        (6001,'/c/en/apple','apple','apple_6001',1,3,0,0,'A round fruit with red or green skin'),
        (6002,'/c/en/bread','bread','bread_6002',1,3,0,0,'Food made from flour, water, and yeast'),
        (6003,'/c/en/water','water','water_6003',1,3,0,0,'A transparent liquid essential for life'),
        (6004,'/c/en/milk','milk','milk_6004',1,3,0,0,'White liquid produced by mammals'),
        (6005,'/c/en/ice','ice','ice_6005',1,3,0,0,'Frozen water; a solid form of water'),
        (6006,'/c/en/cheese','cheese','cheese_6006',1,3,0,0,'A food made from pressed milk curds'),
        (6007,'/c/en/food','food','food_6007',1,3,0,0,'Any substance consumed for nutrition'),
        (6008,'/c/en/plant','plant','plant_6008',1,1,3,0,'A living organism that grows in the earth'),
        (6009,'/c/en/fruit','fruit','fruit_6009',1,3,0,0,'The sweet product of a tree or plant'),
        # Abstract
        (7001,'/c/en/love','love','love_7001',2,0,0,0,'An intense feeling of deep affection'),
        (7002,'/c/en/fear','fear','fear_7002',2,0,0,0,'An unpleasant emotion caused by threat'),
        (7003,'/c/en/knowledge','knowledge','knowledge_7003',2,0,0,0,'Facts or information acquired through experience'),
        # Celestial / Natural
        (8001,'/c/en/sun','sun','sun_8001',1,5,0,0,'The star at the center of the solar system'),
        (8002,'/c/en/sky','sky','sky_8002',1,5,0,0,'The region of the atmosphere seen from earth'),
        (8003,'/c/en/night','night','night_8003',1,5,0,0,'The period of darkness between sunset and sunrise'),
        (8004,'/c/en/fire','fire','fire_8004',1,5,0,0,'Combustion producing heat, light, and flame'),
        (8005,'/c/en/snow','snow','snow_8005',1,5,0,0,'Frozen atmospheric water vapor falling as white flakes'),
        (8006,'/c/en/morning','morning','morning_8006',1,5,0,0,'The early part of the day'),
        (8007,'/c/en/evening','evening','evening_8007',1,5,0,0,'The later part of the day'),
        (8008,'/c/en/star','star','star_8008',1,5,0,0,'A luminous celestial body'),
        (8009,'/c/en/rain','rain','rain_8009',1,5,0,0,'Water falling from clouds in drops'),
        # Plants
        (9001,'/c/en/grass','grass','grass_9001',1,1,3,0,'Green vegetation covering the ground'),
        (9002,'/c/en/tree','tree','tree_9002',1,1,3,0,'A tall perennial plant with a woody trunk'),
    ]
    cur.executemany(
        'INSERT OR IGNORE INTO concepts (id,uri,surface_text,anchor_token,domain,category,subcategory,specificity,definition) VALUES (?,?,?,?,?,?,?,?,?)',
        concepts)

    relations = [
        # IsA (1) — Proper taxonomy
        (1001,1009,1,0.99),(1001,1011,1,0.95),(1001,1012,1,0.90),
        (1002,1009,1,0.99),(1002,1011,1,0.95),(1002,1012,1,0.90),
        (1003,1009,1,0.99),(1004,1009,1,0.99),
        (1005,1003,1,0.95),(1005,1009,1,0.99),
        (1006,1010,1,0.95),(1006,1009,1,0.99),
        (1007,1011,1,0.95),(1007,1009,1,0.99),
        (1008,1011,1,0.95),(1008,1009,1,0.99),
        (8001,8008,1,0.95),(6001,6009,1,0.95),
        (9001,6008,1,0.90),(9002,6008,1,0.90),
        # HasA (3) — Body parts
        (1001,1201,3,0.95),(1001,1202,3,0.95),(1001,1203,3,0.90),(1001,1205,3,0.90),
        (1002,1201,3,0.95),(1002,1202,3,0.95),(1002,1205,3,0.90),
        (1003,1204,3,0.95),(1003,1201,3,0.90),(1004,1208,3,0.90),
        (1005,1204,3,0.90),(1006,1208,3,0.90),
        (1007,1201,3,0.95),(1007,1207,3,0.95),(1007,1203,3,0.90),
        (1008,1202,3,0.90),(9002,1206,3,0.90),
        # CapableOf (5)
        (1001,5006,5,0.95),(1001,5002,5,0.90),(1001,5005,5,0.70),(1001,5008,5,0.95),
        (1001,5009,5,0.90),(1001,5010,5,0.95),(1001,5013,5,0.85),(1001,5011,5,0.85),
        (1001,5003,5,0.95),(1001,5020,5,0.90),
        (1002,5007,5,0.95),(1002,5002,5,0.85),(1002,5008,5,0.95),(1002,5012,5,0.85),
        (1002,5014,5,0.90),(1002,5013,5,0.80),(1002,5010,5,0.90),(1002,5003,5,0.95),
        (1003,5004,5,0.95),(1003,5015,5,0.85),(1003,5019,5,0.80),(1003,5010,5,0.85),
        (1004,5005,5,0.98),(1004,5003,5,0.90),
        (1005,5005,5,0.95),(1005,5010,5,0.90),(1005,5003,5,0.90),
        (1006,5005,5,0.70),(1006,5003,5,0.90),
        (1007,5010,5,0.95),(1007,5005,5,0.60),(1007,5003,5,0.95),(1007,5021,5,0.90),
        (1008,5002,5,0.80),(1008,5012,5,0.75),(1008,5003,5,0.90),(1008,5005,5,0.60),
        (1101,5022,5,0.95),(1101,5002,5,0.90),(1101,5010,5,0.95),
        (1101,5003,5,0.95),(1101,5021,5,0.95),(1101,5023,5,0.90),
        (1102,5009,5,0.95),(1102,5024,5,0.90),(1102,5022,5,0.85),
        # AtLocation (6)
        (1001,4001,6,0.80),(1001,2004,6,0.85),(1002,2004,6,0.90),(1002,4002,6,0.70),
        (1003,4001,6,0.75),(1003,8002,6,0.85),(1003,9002,6,0.80),(1003,4009,6,0.85),
        (1004,6003,6,0.95),(1004,4004,6,0.90),(1004,4008,6,0.85),(1004,4007,6,0.85),(1004,4011,6,0.80),
        (1102,4003,6,0.90),(2005,4003,6,0.85),(2006,4002,6,0.80),(2007,4002,6,0.75),
        (1005,4004,6,0.85),(1005,6005,6,0.80),
        (1006,4005,6,0.75),(1006,4010,6,0.80),
        (1007,4001,6,0.50),(1007,4010,6,0.80),
        (1008,2004,6,0.75),(1008,4005,6,0.70),
        (9001,4005,6,0.90),(9001,4001,6,0.85),
        (6001,9002,6,0.80),(1101,2004,6,0.90),(6002,4002,6,0.80),
        # HasProperty (4)
        (1001,3005,4,0.70),(1001,3203,4,0.75),(1001,3414,4,0.85),(1001,3415,4,0.85),
        (1002,3102,4,0.70),(1002,3416,4,0.80),(1002,3418,4,0.75),
        (1003,3402,4,0.80),
        (1005,3007,4,0.70),(1005,3006,4,0.70),(1005,3417,4,0.75),
        (1006,3408,4,0.85),
        (1007,3101,4,0.95),(1007,3401,4,0.95),(1007,3204,4,0.80),
        (1008,3102,4,0.90),(1008,3402,4,0.85),(1008,3411,4,0.75),(1008,3203,4,0.70),
        (6001,3001,4,0.80),(6001,3003,4,0.70),(6001,3405,4,0.85),(6001,3407,4,0.85),
        (6002,3403,4,0.80),
        (6003,3202,4,0.60),(6003,3413,4,0.85),
        (6004,3006,4,0.85),(6004,3420,4,0.75),
        (6005,3202,4,0.98),(6005,3006,4,0.80),(6005,3404,4,0.85),
        (8001,3004,4,0.95),(8001,3201,4,0.98),(8001,3301,4,0.95),
        (8002,3002,4,0.95),
        (8003,3302,4,0.95),(8003,3411,4,0.80),
        (8004,3001,4,0.90),(8004,3201,4,0.98),(8004,3301,4,0.90),(8004,3422,4,0.90),
        (8005,3006,4,0.98),(8005,3202,4,0.95),(8005,3403,4,0.80),
        (9001,3003,4,0.95),(9001,3403,4,0.75),
        (9002,3003,4,0.80),(9002,3101,4,0.70),(9002,3409,4,0.80),
        (4004,3002,4,0.85),(4004,3410,4,0.85),(4004,3406,4,0.80),(4004,3101,4,0.80),(4004,3421,4,0.70),
        (4001,3003,4,0.80),(2001,3407,4,0.90),(2005,3423,4,0.80),
        # Antonym (22) — bidirectional
        (3201,3202,22,0.95),(3202,3201,22,0.95),
        (3101,3102,22,0.95),(3102,3101,22,0.95),
        (3203,3204,22,0.95),(3204,3203,22,0.95),
        (3302,3301,22,0.95),(3301,3302,22,0.95),
        (3205,3424,22,0.90),(3424,3205,22,0.90),
        (3401,3402,22,0.95),(3402,3401,22,0.95),
        (3403,3404,22,0.95),(3404,3403,22,0.95),
        (3411,3412,22,0.95),(3412,3411,22,0.95),
        # UsedFor (12)
        (2001,5009,12,0.90),(2005,7003,12,0.85),(2003,5002,12,0.60),
        (2006,5003,12,0.70),(2007,5001,12,0.90),
        (2004,5008,12,0.80),(4003,5024,12,0.90),(4002,5023,12,0.85),
        (4001,5009,12,0.85),(6003,5021,12,0.90),
        # MadeOf (16)
        (6002,6003,16,0.60),(6005,6003,16,0.95),(8005,6003,16,0.90),
        # Causes (7)
        (7002,5002,7,0.60),(8004,7002,7,0.50),(8005,3202,7,0.70),
        (3201,5017,7,0.85),(3202,5018,7,0.85),(8001,3301,7,0.80),(8009,6003,7,0.75),
        # Desires (18)
        (1001,5009,18,0.85),(1001,6007,18,0.90),
        (1002,1004,18,0.70),(1002,6007,18,0.85),(1002,1008,18,0.75),
        (1008,6006,18,0.80),(1007,6008,18,0.85),(1102,5009,18,0.90),
        # LocatedNear (29)
        (9001,9002,29,0.80),(4007,4010,29,0.75),
    ]
    cur.executemany('INSERT INTO relations (source_id,target_id,relation_type_id,weight) VALUES (?,?,?,?)', relations)

    conn.commit()
    nc = cur.execute('SELECT COUNT(*) FROM concepts').fetchone()[0]
    nr = cur.execute('SELECT COUNT(*) FROM relations').fetchone()[0]
    conn.close()
    print(f'Micro Bible built: {nc} concepts, {nr} relations at {db_path}')


# ── Full Bible Builder (ConceptNet 5.7 + WordNet) ─────────────────

import csv
import gzip
import urllib.request

CONCEPTNET_URL = 'https://s3.amazonaws.com/conceptnet/downloads/2019/edges/conceptnet-assertions-5.7.0.csv.gz'
CONCEPTNET_MIN_WEIGHT = 1.0


def _parse_conceptnet_uri(uri):
    """Extract English surface text from a ConceptNet URI like /c/en/dog."""
    parts = uri.split('/')
    if len(parts) >= 4 and parts[2] == 'en':
        return parts[3].replace('_', ' ')
    return None


def _download_conceptnet(cache_path):
    """Download ConceptNet CSV dump if not already cached."""
    cache_path = Path(cache_path)
    if cache_path.exists():
        print(f'Using cached ConceptNet dump: {cache_path}')
        return cache_path
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    print(f'Downloading ConceptNet from {CONCEPTNET_URL}...')
    print('This is a large file (~500MB) and may take a while.')
    urllib.request.urlretrieve(CONCEPTNET_URL, str(cache_path))
    print(f'Downloaded to {cache_path}')
    return cache_path


def _get_wordnet_taxonomy(surface_text):
    """Get WordNet taxonomy info for a surface text."""
    try:
        from nltk.corpus import wordnet as wn
    except ImportError:
        return {'domain': 0, 'category': 0, 'subcategory': 0, 'specificity': 0}

    synsets = wn.synsets(surface_text.replace(' ', '_'))
    if not synsets:
        return {'domain': 0, 'category': 0, 'subcategory': 0, 'specificity': 0}

    synset = synsets[0]
    pos = synset.pos()

    domain = 1  # default: physical
    if pos == 'v':
        domain = 3  # action
    elif pos in ('a', 's', 'r'):
        domain = 4  # property

    hypernyms = synset.hypernym_paths()
    if not hypernyms:
        return {'domain': domain, 'category': 0, 'subcategory': 0, 'specificity': 0}

    chain = hypernyms[0]
    category = 0
    subcategory = 0

    top_level_map = {
        'entity.n.01': 0, 'physical_entity.n.01': 1, 'abstraction.n.06': 2,
        'thing.n.12': 1, 'object.n.01': 2, 'whole.n.02': 2,
        'living_thing.n.01': 1, 'organism.n.01': 1, 'animal.n.01': 2,
        'plant.n.02': 3, 'person.n.01': 1, 'artifact.n.01': 2,
        'food.n.01': 3, 'substance.n.01': 3, 'location.n.01': 4,
    }

    for ancestor in chain:
        name = ancestor.name()
        if name in top_level_map:
            if category == 0:
                category = top_level_map[name]
            elif subcategory == 0:
                subcategory = top_level_map[name]

    specificity = min(len(chain) - 1, 5) if len(chain) > 1 else 0
    return {'domain': domain, 'category': category, 'subcategory': subcategory, 'specificity': specificity}


_PROPERTY_CATEGORY_SEEDS = {
    'red': 1, 'blue': 1, 'green': 1, 'yellow': 1, 'brown': 1,
    'white': 1, 'black': 1, 'orange': 1, 'purple': 1, 'pink': 1,
    'gray': 1, 'grey': 1, 'golden': 1, 'silver': 1, 'crimson': 1,
    'scarlet': 1, 'violet': 1, 'maroon': 1, 'turquoise': 1, 'beige': 1,
    'tan': 1, 'ivory': 1, 'indigo': 1, 'magenta': 1, 'teal': 1,
    'colorful': 1, 'colourful': 1, 'multicolored': 1,
    'big': 2, 'small': 2, 'tall': 2, 'deep': 2, 'large': 2,
    'tiny': 2, 'huge': 2, 'short': 2, 'narrow': 2, 'wide': 2,
    'vast': 2, 'little': 2, 'enormous': 2, 'giant': 2, 'massive': 2,
    'miniature': 2, 'petite': 2, 'compact': 2, 'immense': 2, 'thick': 2, 'thin': 2,
    'hot': 3, 'cold': 3, 'warm': 3, 'cool': 3, 'freezing': 3,
    'boiling': 3, 'chilly': 3, 'icy': 3, 'lukewarm': 3, 'tepid': 3, 'scorching': 3,
    'fast': 4, 'slow': 4, 'quick': 4, 'rapid': 4, 'swift': 4, 'speedy': 4, 'sluggish': 4,
    'old': 5, 'new': 5, 'young': 5, 'ancient': 5, 'modern': 5,
    'fresh': 5, 'elderly': 5, 'youthful': 5, 'aged': 5, 'juvenile': 5,
    'bright': 6, 'dark': 6, 'dim': 6, 'shiny': 6, 'dull': 6,
    'glowing': 6, 'radiant': 6, 'luminous': 6, 'brilliant': 6, 'faint': 6,
    'heavy': 7, 'light': 7, 'lightweight': 7, 'weightless': 7,
    'soft': 8, 'hard': 8, 'rough': 8, 'smooth': 8, 'fuzzy': 8,
    'coarse': 8, 'silky': 8, 'bumpy': 8, 'fluffy': 8, 'rigid': 8, 'tender': 8, 'crispy': 8,
    'sweet': 9, 'salty': 9, 'bitter': 9, 'sour': 9, 'spicy': 9,
    'bland': 9, 'savory': 9, 'tangy': 9, 'delicious': 9, 'tasty': 9,
    'round': 10, 'long': 10, 'flat': 10, 'curved': 10, 'square': 10,
    'circular': 10, 'oval': 10, 'straight': 10, 'triangular': 10, 'cylindrical': 10, 'spherical': 10,
    'quiet': 11, 'loud': 11, 'noisy': 11, 'silent': 11, 'deafening': 11, 'mute': 11,
    'clear': 12, 'transparent': 12, 'opaque': 12, 'murky': 12, 'foggy': 12, 'cloudy': 12,
    'friendly': 13, 'loyal': 13, 'independent': 13, 'cute': 13, 'brave': 13,
    'gentle': 13, 'shy': 13, 'kind': 13, 'mean': 13, 'cruel': 13,
    'honest': 13, 'generous': 13, 'stubborn': 13, 'playful': 13, 'aggressive': 13,
    'calm': 13, 'nervous': 13, 'lazy': 13, 'curious': 13, 'clever': 13,
}

_DEFINITION_HINTS = {
    1: ['color', 'colour', 'hue', 'pigment', 'chromatic'],
    2: ['size', 'extent', 'dimension', 'stature', 'height'],
    3: ['temperature', 'thermal', 'heat', 'warmth', 'cold'],
    4: ['speed', 'velocity', 'pace', 'motion'],
    6: ['light', 'luminous', 'bright', 'illuminat'],
    7: ['weight', 'mass', 'heaviness'],
    8: ['texture', 'surface', 'touch', 'tactile'],
    9: ['taste', 'flavor', 'palate'],
    10: ['shape', 'form', 'geometry', 'contour'],
    11: ['sound', 'noise', 'auditory', 'acoustic'],
    12: ['clarity', 'transparent', 'visibility', 'optic'],
}


def _classify_property_category(surface_text):
    """Classify a property-domain concept into a specific category (1-13) or 0."""
    cat = _PROPERTY_CATEGORY_SEEDS.get(surface_text.lower())
    if cat is not None:
        return cat
    try:
        from nltk.corpus import wordnet as wn
        synsets = wn.synsets(surface_text.replace(' ', '_'))
        if synsets:
            defn = synsets[0].definition().lower()
            for cat_id, keywords in _DEFINITION_HINTS.items():
                if any(kw in defn for kw in keywords):
                    return cat_id
    except Exception:
        pass
    return 0


def _generate_anchor_token(surface_text, concept_id):
    """Generate string-anchored token like dog_1001."""
    clean = surface_text.lower().replace(' ', '_').replace('-', '_')
    clean = ''.join(c for c in clean if c.isalnum() or c == '_')
    return f'{clean}_{concept_id}'


def build_full_bible(db_path, conceptnet_cache=None, max_concepts=100000, progress_interval=10000):
    """Build the full SML Bible from ConceptNet 5.7 + WordNet.

    Downloads ConceptNet (~500MB), extracts English concepts/relations,
    enriches with WordNet taxonomy, and builds the SQLite database.
    """
    from tqdm.notebook import tqdm

    db_path = str(db_path)
    if conceptnet_cache is None:
        conceptnet_cache = str(Path(db_path).parent / 'conceptnet-assertions-5.7.0.csv.gz')

    cn_path = _download_conceptnet(Path(conceptnet_cache))
    create_bible_db(db_path)
    conn = sqlite3.connect(db_path)

    # Phase 1: Parse ConceptNet
    print('Phase 1: Parsing ConceptNet assertions...')
    concepts = {}
    raw_relations = []

    rel_uri_to_id = {
        '/r/IsA': 1, '/r/PartOf': 2, '/r/HasA': 3, '/r/HasProperty': 4,
        '/r/CapableOf': 5, '/r/AtLocation': 6, '/r/Causes': 7,
        '/r/HasPrerequisite': 8, '/r/HasFirstSubevent': 9,
        '/r/HasLastSubevent': 10, '/r/MotivatedByGoal': 11, '/r/UsedFor': 12,
        '/r/CreatedBy': 13, '/r/DefinedAs': 14, '/r/SymbolOf': 15,
        '/r/MadeOf': 16, '/r/ReceivesAction': 17, '/r/Desires': 18,
        '/r/CausesDesire': 19, '/r/HasContext': 20, '/r/SimilarTo': 21,
        '/r/Antonym': 22, '/r/DerivedFrom': 23, '/r/RelatedTo': 24,
        '/r/FormOf': 25, '/r/EtymologicallyRelatedTo': 26, '/r/Synonym': 27,
        '/r/MannerOf': 28, '/r/LocatedNear': 29,
        '/r/dbpedia/genre': 31, '/r/dbpedia/occupation': 32,
        '/r/dbpedia/language': 33, '/r/dbpedia/capital': 34,
    }

    concept_id_counter = 1

    with gzip.open(str(cn_path), 'rt', encoding='utf-8') as f:
        reader = csv.reader(f, delimiter='	')
        for row in tqdm(reader, desc='Reading ConceptNet', unit=' rows'):
            if len(row) < 5:
                continue
            rel_uri, source_uri, target_uri = row[1], row[2], row[3]
            try:
                meta = json.loads(row[4])
                weight = meta.get('weight', 0.0)
            except (json.JSONDecodeError, IndexError):
                weight = 0.0

            if weight < CONCEPTNET_MIN_WEIGHT:
                continue

            source_text = _parse_conceptnet_uri(source_uri)
            target_text = _parse_conceptnet_uri(target_uri)
            if source_text is None or target_text is None:
                continue

            rel_id = rel_uri_to_id.get(rel_uri)
            if rel_id is None:
                continue

            for text, uri in [(source_text, source_uri), (target_text, target_uri)]:
                if text not in concepts and len(concepts) < max_concepts:
                    concepts[text] = {'uri': uri, 'id': concept_id_counter, 'surface_text': text}
                    concept_id_counter += 1

            if source_text in concepts and target_text in concepts:
                raw_relations.append((source_text, target_text, rel_id, min(weight / 10.0, 1.0)))

    print(f'Extracted {len(concepts)} unique concepts and {len(raw_relations)} relations')

    # Phase 2: Enrich with WordNet taxonomy
    print('Phase 2: Enriching with WordNet taxonomy...')
    concept_rows = []
    for i, (text, info) in enumerate(concepts.items()):
        if i > 0 and i % progress_interval == 0:
            print(f'  Processed {i}/{len(concepts)} concepts...')

        taxonomy = _get_wordnet_taxonomy(text)
        if taxonomy['domain'] == 4:
            taxonomy['category'] = _classify_property_category(text)

        anchor = _generate_anchor_token(text, info['id'])

        definition = ''
        try:
            from nltk.corpus import wordnet as wn
            synsets = wn.synsets(text.replace(' ', '_'))
            if synsets:
                defn = synsets[0].definition()
                if defn:
                    definition = defn
        except Exception:
            pass

        concept_rows.append((
            info['id'], info['uri'], text, anchor,
            taxonomy['domain'], taxonomy['category'],
            taxonomy['subcategory'], taxonomy['specificity'],
            definition,
        ))

    # Phase 3: Insert concepts
    print('Phase 3: Inserting concepts into database...')
    conn.executemany(
        'INSERT OR IGNORE INTO concepts (id, uri, surface_text, anchor_token, domain, category, subcategory, specificity, definition) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)',
        concept_rows,
    )

    # Phase 4: Insert relations
    print('Phase 4: Inserting relations...')
    relation_rows = []
    for source_text, target_text, rel_id, weight in raw_relations:
        if source_text in concepts and target_text in concepts:
            relation_rows.append((concepts[source_text]['id'], concepts[target_text]['id'], rel_id, weight))

    conn.executemany(
        'INSERT INTO relations (source_id, target_id, relation_type_id, weight) VALUES (?, ?, ?, ?)',
        relation_rows,
    )
    conn.commit()

    # Phase 5: Rebuild FTS index
    print('Phase 5: Rebuilding FTS index...')
    conn.execute("INSERT INTO concepts_fts(concepts_fts) VALUES('rebuild')")
    conn.commit()
    conn.close()

    print(f'\nFull Bible built: {len(concepts)} concepts, {len(relation_rows)} relations at {db_path}')


# ── Build Bible based on RUN_MODE ─────────────────────────────────
if RUN_MODE == 'test':
    print('Building MICRO Bible (test mode)...')
    build_micro_bible(str(BIBLE_DB_PATH))
else:
    print('Building FULL Bible from ConceptNet + WordNet...')
    build_full_bible(str(BIBLE_DB_PATH))

## 3. Bible Query Interface

In [ ]:
class Bible:
    """Read-only query interface to the SML Bible."""
    def __init__(self, db_path):
        self.conn = sqlite3.connect(db_path)
        self.conn.row_factory = sqlite3.Row

    def close(self):
        self.conn.close()

    def lookup_concept(self, text):
        row = self.conn.execute('SELECT * FROM concepts WHERE LOWER(surface_text) = LOWER(?)', (text,)).fetchone()
        return dict(row) if row else None

    def lookup_by_anchor(self, anchor):
        row = self.conn.execute('SELECT * FROM concepts WHERE anchor_token = ?', (anchor,)).fetchone()
        return dict(row) if row else None

    def get_outgoing_relations(self, concept_id):
        rows = self.conn.execute(
            '''SELECT r.*, rt.label as relation_label, c.surface_text as target_text, c.anchor_token as target_anchor
               FROM relations r JOIN relation_types rt ON r.relation_type_id = rt.id
               JOIN concepts c ON r.target_id = c.id WHERE r.source_id = ?''', (concept_id,)).fetchall()
        return [dict(r) for r in rows]

    def get_concept_by_id(self, concept_id):
        row = self.conn.execute('SELECT * FROM concepts WHERE id = ?', (concept_id,)).fetchone()
        return dict(row) if row else None

    def search_fuzzy(self, text, limit=5):
        rows = self.conn.execute(
            '''SELECT c.* FROM concepts_fts fts JOIN concepts c ON fts.rowid = c.id
               WHERE concepts_fts MATCH ? ORDER BY rank LIMIT ?''', (f'{text}*', limit)).fetchall()
        return [dict(r) for r in rows]

# Quick validation
bible = Bible(str(BIBLE_DB_PATH))
sun = bible.lookup_concept('sun')
sun_props = [r['target_text'] for r in bible.get_outgoing_relations(sun['id']) if r['relation_type_id'] == 4]
print(f'sun properties: {sun_props}')
bible.close()

## 4. SML Formatter (with NOT_ negation)

In [ ]:
def format_eda(array):
    if len(array) != 8: raise ValueError(f'EDA must have 8 elements, got {len(array)}')
    return f"E({array[0]}|{array[1]}|{array[2]}|{array[3]}|{array[4]}|{array[5]}|{array[6]}|{array[7]})"

def format_ra(array):
    if len(array) != 6: raise ValueError(f'RA must have 6 elements, got {len(array)}')
    rel_type, negation = array[0], array[5]
    label = RELATION_TYPES.get(rel_type, str(rel_type)) if isinstance(rel_type, int) else str(rel_type)
    if negation and str(negation) not in ('0', '0.0', 'False'):
        label = f'NOT_{label}'
    return f"R({label}|{array[1]}|{array[2]}|{array[3]}|{array[4]}|0)"

def format_sml_block(entities, relations):
    lines = [format_eda(e) for e in entities] + [format_ra(r) for r in relations]
    return f"<sml>\n{'\n'.join(lines)}\n</sml>"

def parse_sml_block(sml_text):
    entities, relations = [], []
    content = sml_text.strip()
    if content.startswith('<sml>'): content = content[5:]
    if content.endswith('</sml>'): content = content[:-6]
    for line in content.strip().split('\n'):
        line = line.strip()
        if not line: continue
        if line.startswith('E(') and line.endswith(')'):
            parts = line[2:-1].split('|')
            parsed = []
            for p in parts:
                try: parsed.append(int(p))
                except ValueError:
                    try: parsed.append(float(p))
                    except ValueError: parsed.append(p)
            entities.append(parsed)
        elif line.startswith('R(') and line.endswith(')'):
            parts = line[2:-1].split('|')
            parsed, negation = [], 0
            for idx, p in enumerate(parts):
                if idx == 0:
                    rel_label = p
                    if p.startswith('NOT_'): negation = 1; rel_label = p[4:]
                    parsed.append(RELATION_TYPES_INV.get(rel_label, rel_label))
                else:
                    try: parsed.append(int(p))
                    except ValueError:
                        try: parsed.append(float(p))
                        except ValueError: parsed.append(p)
            if len(parsed) == 6: parsed[5] = negation
            relations.append(parsed)
    return {'entities': entities, 'relations': relations}

# Quick test
test_block = format_sml_block(
    [[1,1,2,1,'dog_1001',0,0,0.9]],
    [[5,0,1,0.9,4,0], [5,0,1,0.9,4,1]]
)
print(test_block)
parsed = parse_sml_block(test_block)
print(f'Round-trip negation flags: {parsed["relations"][0][5]}, {parsed["relations"][1][5]}')

## 5. SML Encoder (with Bible modifier query)

In [ ]:
import spacy
nlp = spacy.load('en_core_web_sm')


class SMLEncoder:
    """Encodes natural language text into SML blocks via spaCy + Bible."""

    def __init__(self, bible_path):
        self.bible = Bible(bible_path)

    def close(self):
        self.bible.close()

    def _resolve_concept(self, token_text, context=''):
        concept = self.bible.lookup_concept(token_text.lower())
        if concept: return concept
        doc = nlp(token_text)
        if doc and doc[0].lemma_ != token_text.lower():
            concept = self.bible.lookup_concept(doc[0].lemma_)
            if concept: return concept
        results = self.bible.search_fuzzy(token_text.lower(), limit=3)
        return results[0] if results else None

    def _make_unknown_eda(self, word):
        return [0, 0, 0, 0, f"unknown_{word.lower().replace(' ','_')}", 0, 0, 0.3]

    def _concept_to_eda(self, concept, modifiers=None, confidence=0.9):
        mod1 = modifiers[0]['anchor_token'] if modifiers and len(modifiers) > 0 else 0
        mod2 = modifiers[1]['anchor_token'] if modifiers and len(modifiers) > 1 else 0
        return [concept['domain'], concept['category'], concept['subcategory'],
                concept['specificity'], concept['anchor_token'], mod1, mod2, round(confidence, 2)]

    def _get_bible_modifiers(self, concept, text, max_modifiers=2):
        """Query Bible HasProperty relations for modifier slots."""
        rels = self.bible.get_outgoing_relations(concept['id'])
        has_prop = [r for r in rels if r['relation_type_id'] == 4]
        if not has_prop: return []
        has_prop.sort(key=lambda r: r['weight'], reverse=True)

        text_lower = text.lower()
        priority_cat = None
        if any(w in text_lower for w in ('color', 'colour')): priority_cat = 1
        elif any(w in text_lower for w in ('big', 'small', 'size', 'tall')): priority_cat = 2
        elif any(w in text_lower for w in ('hot', 'cold', 'temperature')): priority_cat = 3
        elif any(w in text_lower for w in ('fast', 'slow', 'speed')): priority_cat = 4
        elif any(w in text_lower for w in ('bright', 'dark')): priority_cat = 6
        elif any(w in text_lower for w in ('heavy', 'light', 'weight')): priority_cat = 7
        elif any(w in text_lower for w in ('soft', 'hard', 'texture')): priority_cat = 8
        elif any(w in text_lower for w in ('sweet', 'salty', 'taste')): priority_cat = 9
        elif any(w in text_lower for w in ('round', 'long', 'shape')): priority_cat = 10
        elif any(w in text_lower for w in ('quiet', 'loud', 'sound', 'noise')): priority_cat = 11
        elif any(w in text_lower for w in ('clear', 'transparent')): priority_cat = 12
        elif any(w in text_lower for w in ('friendly', 'loyal', 'independent', 'cute')): priority_cat = 13

        prioritized, others = [], []
        for rel in has_prop:
            target = self.bible.get_concept_by_id(rel['target_id'])
            if not target: continue
            if priority_cat and target['domain'] == 4 and target['category'] == priority_cat:
                prioritized.append(target)
            else:
                others.append(target)
        return (prioritized + others)[:max_modifiers]

    def encode(self, text):
        doc = nlp(text)
        entities, relations, entity_index = [], [], {}

        for chunk in doc.noun_chunks:
            head_text = chunk.root.lemma_.lower()
            concept = self._resolve_concept(head_text, text)
            modifiers = []
            for token in chunk:
                if token.dep_ in ('amod', 'acomp') and token != chunk.root:
                    mod_c = self._resolve_concept(token.lemma_.lower(), text)
                    if mod_c: modifiers.append(mod_c)
            if concept:
                if not modifiers:
                    modifiers = self._get_bible_modifiers(concept, text)
                eda = self._concept_to_eda(concept, modifiers)
            else:
                eda = self._make_unknown_eda(head_text)
            idx = len(entities)
            entities.append(eda)
            entity_index[chunk.root.i] = idx

        for token in doc:
            if token.pos_ == 'VERB':
                subj_idx = obj_idx = None
                for child in token.children:
                    if child.dep_ in ('nsubj', 'nsubjpass') and child.i in entity_index:
                        subj_idx = entity_index[child.i]
                    elif child.dep_ in ('dobj', 'attr') and child.i in entity_index:
                        obj_idx = entity_index[child.i]
                if obj_idx is None:
                    for child in token.children:
                        if child.dep_ == 'prep':
                            for gc in child.children:
                                if gc.dep_ == 'pobj' and gc.i in entity_index:
                                    obj_idx = entity_index[gc.i]; break
                if subj_idx is not None and obj_idx is not None:
                    rel_type = 24
                    for child in token.children:
                        if child.dep_ == 'prep' and child.text.lower() in ('on','in','at','near','under'):
                            rel_type = 6; break
                    verb_map = {'be':1,'have':3,'cause':7,'use':12,'make':13,'want':18,'need':8,'eat':12}
                    vl = token.lemma_.lower()
                    if vl in verb_map: rel_type = verb_map[vl]
                    temporal = 0
                    if token.morph.get('Tense'):
                        t = token.morph.get('Tense')[0]
                        temporal = 1 if t == 'Past' else 2 if t == 'Pres' else 0
                    relations.append([rel_type, subj_idx, obj_idx, 0.85, temporal, 0])

        if not entities:
            for token in doc:
                if token.pos_ in ('NOUN', 'PROPN'):
                    concept = self._resolve_concept(token.lemma_.lower(), text)
                    entities.append(self._concept_to_eda(concept) if concept else self._make_unknown_eda(token.lemma_.lower()))
        if not entities:
            entities.append(self._make_unknown_eda(text[:20]))

        return format_sml_block(entities, relations)


# Quick test
enc = SMLEncoder(str(BIBLE_DB_PATH))
print(enc.encode('What color is the sun?'))
print()
print(enc.encode('Can penguins fly?'))
print()
print(enc.encode('Is snow soft?'))
enc.close()

## 6. Generate Training Data (Groq)

This calls the Groq API to generate `<think>` + `<response>` for each prompt.

**Takes ~15-20 min for 1500 examples** (rate-limited to ~30 req/min on free tier).
**Takes ~2-3 hours for 15K examples** — use a Colab Pro/A100 runtime for full runs.

In [ ]:
# ── Prompt bank (349 prompts) ─────────────────────────────────────
# Trimmed inline for notebook — full list from data_generator.py

MICRO_PROMPTS = [
    # Factual / Science
    'What color is the sun?', 'What color is the sky?', 'What color is grass?',
    'What color is fire?', 'What color is snow?', 'What color are apples?',
    'What color is the ocean?', 'What color is milk?', 'What color can a dog be?',
    'Is the sun hot or cold?', 'Is fire hot?', 'Is snow cold?', 'Is ice cold?',
    'Is water hot or cold?', 'What is ice made of?', 'What is snow made of?',
    'What is bread made of?', 'Is the sun bright?', 'Is the night dark?',
    'Can birds fly?', 'Can fish swim?', 'Can dogs swim?', 'Can penguins swim?',
    'Can elephants walk?', 'Can cats climb?', 'Can mice climb?', 'Can snakes swim?',
    'Can dogs bark?', 'Can cats purr?', 'Can dogs run fast?',
    'What does the sun look like?', 'Is an apple a fruit?', 'Is a bird a type of animal?',
    'What color can apples be?', 'Is the sky blue during the day?',
    'Does grass grow on the ground?', 'Is fire dangerous?',
    'Does snow fall from the sky?', 'What color is a tree?', 'Are trees green?',
    'Is the ocean big?', 'Is the ocean blue?', 'Are elephants heavy?',
    'Are mice light?', 'Is an elephant big or small?', 'Is a mouse big or small?',
    'What color is a penguin?', 'Do dogs have four legs?', 'Can a person feel fear?',
    'What is love?', 'What is knowledge?', 'What animals can swim?',
    'What animals bark?', 'What animals can fly?', 'Do fish live in water?',
    'What do dogs like to do?', 'What do cats do when they are happy?',
    'What is milk?', 'How heavy is an elephant?', 'How small is a mouse?',
    'Is the night sky dark?', 'What does fire look like?',
    'Is ice the same as water?', 'Can dogs hear well?', 'What color is night?',
    'Is fire bright?', 'Is the sun yellow?', 'Is grass green?',
    'Is the sky usually blue?', 'Do elephants swim?', 'Can mice run?',
    'Do cats sleep a lot?', 'Do dogs sleep?', 'Can birds swim?',
    'Are penguins birds?', 'Is a snake a reptile?', 'Do mice eat cheese?',
    'Is snow white?', 'Is ice transparent?', 'Can elephants climb trees?',
    'Are trees big?', 'Do trees have leaves?', 'Is the sun a star?',
    'Is water blue?', 'Is fire red?', 'What temperature is ice?',
    'What temperature is fire?', 'Is the ocean cold?', 'Is milk white?',
    'Do penguins walk?',

    # Commonsense
    'Where do dogs like to go?', 'Where do cats live?', 'Where do fish live?',
    'Where do children go during the day?', 'What do children like to do?',
    'What is a chair used for?', 'What is a book used for?',
    'What are houses used for?', 'What is a table used for?',
    'What is a ball used for?', 'Can a car fly?', 'What do you need water for?',
    'What do you feel when something scary happens?',
    'What is the opposite of hot?', 'What do dogs eat?',
    'What do cats like to eat?', 'Where does a dog sleep?',
    'What does a child do at school?', 'Why do dogs wag their tails?',
    'Why do cats purr?', 'What happens when you throw a ball?',
    'What can you do at a park?', 'What do you do when you are hungry?',
    'What do you do when you are sleepy?', 'What can you find in a kitchen?',
    'What is a car used for?', 'Why do people read books?',
    'What is the opposite of cold?', 'What is the opposite of big?',
    'What is the opposite of fast?', 'What is the opposite of dark?',
    'What is the opposite of old?', 'Do people live in houses?',
    'Do fish need water?', 'Do dogs need food?', 'Do birds build nests?',
    'What do people do in the kitchen?', 'Where do fish swim?',
    'Where do birds fly?', 'Do children play?', 'Do dogs play?',
    'What does a cat do at home?', 'What does a dog do at the park?',
    'What is a park for?', 'What is a school for?',
    'Why do people drink water?', 'Can a fish live on land?',
    'Can a bird walk?', 'Do penguins like cold weather?',
    'Do elephants eat plants?', 'What do mice like to eat?',
    'Where do snakes live?', 'Where do penguins live?',
    'Where do elephants live?', 'Do dogs chase cats?',
    'Do cats chase mice?', 'What happens when a dog sees a ball?',
    'What happens when a cat sees a mouse?', 'Why do fish swim?',
    'Why do birds fly?', 'Why do dogs bark?', 'What sound does a dog make?',
    'What sound does a cat make?', 'Is an elephant a good pet?',
    'Is a mouse a big animal?', 'What makes a dog happy?',
    'What makes a cat happy?', 'Do children go to school?',
    'Do dogs go to school?', 'What does a bird do in the morning?',
    'What does a child do in the evening?', 'Where can you find a chair?',
    'Where can you find a table?', 'What does a person do with a book?',

    # Spatial / Location
    'The dog sat on the mat. Where is the dog?',
    'The cat sat on the chair. Where is the cat?',
    'The child is at school. What is the child doing?',
    'The red ball is in the park. What color is the ball?',
    'The big brown dog ran in the park. Describe the scene.',
    'The small black cat is sleeping. What is the cat doing?',
    'The bird flew over the house. Can birds fly?',
    'Where might you find a table?', 'Where might you find a book?',
    'The fish is in the ocean. Where is the fish?',
    'The elephant is walking in the park. What is it doing?',
    'The penguin is swimming in the ocean. Describe the scene.',
    'A child is reading a book at school. Where is the child?',
    'The dog is sleeping in the house. Where is the dog?',
    'The cat is in the kitchen. Where is the cat?',
    'The ball is under the table. Where is the ball?',
    'The bird is in the tree. Where is the bird?',
    'There is snow on the ground. What does the ground look like?',
    'The sun is in the sky. Where is the sun?',
    'The fire is in the kitchen. Where is the fire?',
    'A mouse is in the house. Where is the mouse?',
    'The snake is near the tree. Where is the snake?',
    'The chair is in the kitchen. Where is the chair?',
    'A dog and a cat are in the house. Where are the animals?',
    'The child is playing in the park. What is the child doing?',
    'The fish lives in a pond. Where is the fish?',
    'The book is on the table. Where is the book?',
    'There is ice on the lake. What is on the lake?',
    'The elephant is near the water. What is the elephant near?',
    'The penguin is on the ice. Where is the penguin?',
    'A dog is running in the park. Describe the scene.',
    'The cat is sleeping on the mat. Where is the cat?',
    'The apple is on the table. Where is the apple?',
    'There is bread in the kitchen. Where is the bread?',
    'The child walked to school. Where did the child go?',
    'The bird landed on the tree. Where is the bird?',
    'A big elephant is near the tree. Describe the scene.',
    'The fast dog ran past the house. What happened?',
    'The cold water is in a glass. Describe the water.',
    'The white snow covers the ground. What does it look like?',
    'The green grass is in the park. What color is the grass?',
    'The bright sun is in the sky. Describe the sun.',
    'The dark night sky has no sun. Describe the sky.',
    'A yellow ball is in the park. What color is the ball?',
    'The old book is on the table. Describe the book.',
    'The small mouse is under the chair. Where is the mouse?',
    'The big tree is in the park. Describe the tree.',
    'A brown dog is at the park. Describe the dog.',
    'The hot fire is in the kitchen. What is the fire like?',
    'The cold ice is on the table. What is on the table?',
    'A sleeping cat is on the chair. What is the cat doing?',
    'The fast bird flew over the park. What happened?',
    'A child and a dog are playing at the park. Describe the scene.',
    'The slow elephant walked through the park. What happened?',
    'The white milk is on the table. What color is the milk?',
    'There is a red apple and a green apple. What colors are the apples?',
    'The fish is swimming in the blue ocean. Where is the fish?',
    'The dog is barking at the park. What is the dog doing?',
    'A cat is climbing a tree. What is the cat doing?',
    'The mouse is running under the table. Where is the mouse?',

    # Causation
    'What causes fear?', 'What happens when it snows?', 'What does fear cause?',
    'What happens when fire starts?', 'Why do people run when scared?',
    'What causes ice to melt?', 'What happens when water freezes?',
    'What does snow cause?', 'What happens when the sun comes out?',
    'Why does ice feel cold?', 'Why is fire hot?', 'What causes snow to melt?',
    'What happens when a dog sees food?', 'What causes a dog to bark?',
    'What happens when a cat is scared?', 'What causes water to freeze?',
    'Why do people feel cold in snow?', 'What causes the sky to be blue?',
    'What happens when you pet a cat?', 'What makes a dog want to play?',
    'What causes the night to be dark?', 'Why is snow white?',
    'What happens when the sun goes down?', 'Why do children go to school?',
    'What causes grass to be green?', 'What happens when a ball is thrown?',
    'Why do fish live in water?', 'What causes a person to feel love?',
    'What happens when an elephant walks?', 'Why do penguins swim?',
    'What makes the ocean blue?', 'What happens when a snake sees a mouse?',
    'What causes a bird to sing?', 'Why does the sun look yellow?',
    'What happens when you read a book?', 'Why do cats climb trees?',
    'What causes a person to feel fear?', 'What happens when a mouse sees a cat?',
    'Why do dogs like parks?', 'What causes apples to be red?',
    'What happens when fire meets water?', 'Why is the ocean salty?',
    'What causes snow to fall?', 'What happens when you sit in a chair?',
    'Why do elephants need water?',

    # Properties
    'Is an elephant big?', 'Is a mouse fast?', 'Is a dog friendly?',
    'Is a cat independent?', 'Is the sun far away?', 'Is the ocean deep?',
    'Is snow soft?', 'Is ice hard?', 'Is fire bright?', 'Is the night quiet?',
    'Are trees tall?', 'Is grass soft?', 'Is an elephant slow?',
    'Is a mouse quiet?', 'Is a bird light?', 'Is a fish cold?',
    'Is a penguin cute?', 'Is a snake long?', 'Is a dog loyal?',
    'Is a cat quick?', 'Is the sky big?', 'Is the park green?',
    'Is milk healthy?', 'Is bread soft?', 'Is an apple sweet?',
    'Is water clear?', 'Is fire red or orange?', 'Is the ocean calm?',
    'Is a ball round?', 'Is a book useful?',

    # Negation
    'Can penguins fly?', 'Can fish walk?', 'Can snakes hear?',
    'Is ice hot?', 'Is the night bright?', 'Can elephants fly?',
    'Can mice fly?', 'Is snow hot?', 'Can fish fly?', 'Is the ocean hot?',
    'Can a car swim?', 'Can a table walk?', 'Can a chair fly?',
    'Can a book swim?', 'Is fire cold?', 'Is the sun dark?',
    'Is grass red?', 'Is the sky green?', 'Is milk black?', 'Is snow black?',
    'Can a ball bark?', 'Can a house run?', 'Can a tree fly?',
    'Can water walk?', 'Is ice hot or cold?', 'Can fish walk on land?',
    'Can a dog fly?', 'Can a cat bark?', 'Can a snake fly?',
    'Can an elephant fly?', 'Can a mouse bark?',
    'Is the night bright or dark?', 'Is fire cold or hot?',
    'Is snow warm?', 'Is ice warm?', 'Can a penguin bark?',
    'Can an elephant climb trees?', 'Can a fish run?', 'Can a snake walk?',
    'Is the sun cold?', 'Is water hot?', 'Can a mouse swim?',
    'Can bread fly?', 'Is grass yellow?', 'Is the ocean green?',
    'Is fire white?', 'Can a book walk?', 'Can a chair swim?',
    'Is an elephant small?', 'Is a mouse big?',
]

print(f'Total prompts: {len(MICRO_PROMPTS)}')

In [ ]:
# ── Template-Based Prompt Generator ────────────────────────────────
# Generates thousands of unique prompts by sampling real concept/relation
# pairs from the Bible DB and filling diverse question templates.

import random as _rng_mod
import sqlite3 as _pg_sqlite3

_RELATION_TEMPLATES = {
    4: [  # HasProperty
        "Is {source} {target}?",
        "Would you describe {source} as {target}?",
        "How {target} is {source}?",
        "Describe the {target} quality of {source}.",
        "Can {source} be described as {target}?",
        "In what way is {source} {target}?",
        "Would you say {source} is {target}?",
        "Tell me about how {source} is {target}.",
    ],
    5: [  # CapableOf
        "Can {source} {target}?",
        "Is {source} able to {target}?",
        "What happens when {source} tries to {target}?",
        "Does {source} {target}?",
        "Tell me about {source} and its ability to {target}.",
        "Is it true that {source} can {target}?",
        "How does {source} {target}?",
        "Explain how {source} can {target}.",
    ],
    1: [  # IsA
        "Is {source} a type of {target}?",
        "What kind of {target} is {source}?",
        "Would you classify {source} as a {target}?",
        "Is {source} considered a {target}?",
        "How is {source} related to {target}?",
        "Tell me how {source} is a {target}.",
        "In what sense is {source} a {target}?",
        "Explain the relationship between {source} and {target}.",
    ],
    6: [  # AtLocation
        "Where can you find {source}?",
        "Is {source} found at {target}?",
        "What can you find at {target}?",
        "Where is {source} typically located?",
        "Would you expect to see {source} at {target}?",
        "Tell me where {source} can be found.",
        "Is {target} a place where {source} exists?",
        "Describe where you might encounter {source}.",
    ],
    7: [  # Causes
        "What does {source} cause?",
        "Does {source} lead to {target}?",
        "What happens because of {source}?",
        "How does {source} result in {target}?",
        "Is {target} caused by {source}?",
        "Explain the causal link between {source} and {target}.",
        "What is the effect of {source}?",
        "Tell me about the consequences of {source}.",
    ],
    12: [  # UsedFor
        "What is {source} used for?",
        "Is {source} used for {target}?",
        "What can you do with {source}?",
        "How is {source} typically used?",
        "Can {source} be used to {target}?",
        "Describe a common use of {source}.",
        "Tell me about the purpose of {source}.",
        "What role does {source} play in {target}?",
    ],
    16: [  # MadeOf
        "What is {source} made of?",
        "Is {source} made from {target}?",
        "What material is in {source}?",
        "Does {source} contain {target}?",
        "Tell me what {source} is composed of.",
        "Is {target} a component of {source}?",
        "Describe the composition of {source}.",
        "What are the materials in {source}?",
    ],
    22: [  # Antonym
        "What is the opposite of {source}?",
        "Are {source} and {target} opposites?",
        "How are {source} and {target} different?",
        "Is {target} the opposite of {source}?",
        "Compare {source} and {target}.",
        "Explain the contrast between {source} and {target}.",
        "In what way are {source} and {target} opposed?",
        "Tell me about the difference between {source} and {target}.",
    ],
    3: [  # HasA
        "Does {source} have {target}?",
        "What does {source} have?",
        "Is {target} a part of {source}?",
        "Tell me what {source} has.",
        "Does {source} possess {target}?",
        "What features does {source} have?",
        "Describe what {source} has.",
        "Is {target} something {source} has?",
    ],
    18: [  # Desires
        "What does {source} want?",
        "Does {source} want {target}?",
        "What does {source} desire?",
        "Is {target} something {source} seeks?",
        "Tell me about what {source} wants.",
        "What motivates {source}?",
        "Does {source} seek {target}?",
        "Describe the desires of {source}.",
    ],
}

_PROPERTY_QUERY_TEMPLATES = {
    1: ["What color is {source}?", "Describe the color of {source}.", "What hue is {source}?"],
    2: ["How big is {source}?", "What size is {source}?", "Is {source} large or small?"],
    3: ["Is {source} hot or cold?", "What temperature is {source}?", "How warm is {source}?"],
    4: ["How fast is {source}?", "Is {source} quick or slow?", "What speed does {source} have?"],
    5: ["How old is {source}?", "Is {source} young or old?", "What age is {source}?"],
    6: ["Is {source} bright or dark?", "How bright is {source}?", "Describe the brightness of {source}."],
    7: ["Is {source} heavy or light?", "How much does {source} weigh?", "What is the weight of {source}?"],
    8: ["What texture is {source}?", "Is {source} smooth or rough?", "How does {source} feel to touch?"],
    9: ["How does {source} taste?", "Is {source} sweet or bitter?", "Describe the flavor of {source}."],
    10: ["What shape is {source}?", "Is {source} round or flat?", "Describe the shape of {source}."],
    11: ["Is {source} loud or quiet?", "How noisy is {source}?", "What sound does {source} make?"],
    12: ["Is {source} transparent or opaque?", "How clear is {source}?", "Can you see through {source}?"],
    13: ["Is {source} friendly?", "What personality does {source} have?", "Describe the temperament of {source}."],
}

_SCENE_TEMPLATES_SINGLE = [
    "The {adj} {entity} is at the {location}. Describe the scene.",
    "Imagine a {adj} {entity} at the {location}. What do you see?",
    "A {adj} {entity} sits at the {location}. Tell me about it.",
    "Picture a {adj} {entity} in the {location}. What is happening?",
    "There is a {adj} {entity} at the {location}. Describe it.",
]

_SCENE_TEMPLATES_DOUBLE = [
    "{entity1} and {entity2} are at the {location}. What might happen?",
    "There is a {adj1} {entity1} and a {adj2} {entity2}. Compare them.",
    "A {adj1} {entity1} meets a {adj2} {entity2} at the {location}. Describe the encounter.",
    "{entity1} and {entity2} are both at the {location}. What do they have in common?",
]

_NEGATION_TEMPLATES = [
    "Can {entity} {action}?",
    "Is {entity} able to {action}?",
    "Does {entity} {action}?",
    "Is it true that {entity} can {action}?",
    "Is {entity} {property}?",
    "Would you describe {entity} as {property}?",
    "Can {entity} be considered {property}?",
    "Is it accurate to say {entity} is {property}?",
]

_COMPARISON_TEMPLATES = [
    "Is {entity1} bigger than {entity2}?",
    "Which is faster, {entity1} or {entity2}?",
    "How are {entity1} and {entity2} different?",
    "What do {entity1} and {entity2} have in common?",
    "Compare {entity1} and {entity2}.",
    "Which is heavier, {entity1} or {entity2}?",
    "Is {entity1} more similar to {entity2} or are they very different?",
    "Tell me the differences between {entity1} and {entity2}.",
]

_ENTITY_TEMPLATES = [
    "What is {entity}?",
    "Describe {entity}.",
    "Tell me about {entity}.",
    "What are the properties of {entity}?",
    "What do you know about {entity}?",
    "Explain what {entity} is.",
    "Give me facts about {entity}.",
    "What can you tell me about {entity}?",
]

_TARGET_DISTRIBUTION = {
    "has_property_direct": 0.15,
    "has_property_category": 0.10,
    "capable_of": 0.15,
    "at_location": 0.12,
    "is_a": 0.08,
    "causes": 0.07,
    "used_for": 0.07,
    "scene": 0.08,
    "negation": 0.07,
    "comparison": 0.04,
    "minor_relations": 0.04,
    "entity_only": 0.03,
}


class PromptGenerator:
    """Generate diverse training prompts by sampling concept/relation pairs from the Bible DB."""

    def __init__(self, bible_path, seed=42):
        self.rng = _rng_mod.Random(seed)
        self.conn = _pg_sqlite3.connect(bible_path)
        self._load_data()

    def _load_data(self):
        cur = self.conn.cursor()
        cur.execute("SELECT id, surface_text, domain, category FROM concepts")
        self.concepts = {}
        self.entities_by_domain = {}
        self.properties_by_category = {}
        for row in cur.fetchall():
            cid, text, domain, category = row
            self.concepts[cid] = {"surface_text": text, "domain": domain, "category": category}
            self.entities_by_domain.setdefault(domain, []).append(text)
            if domain == 4:
                self.properties_by_category.setdefault(category, []).append(text)

        cur.execute("""
            SELECT c1.surface_text, c2.surface_text, r.relation_type_id, r.weight
            FROM relations r
            JOIN concepts c1 ON r.source_id = c1.id
            JOIN concepts c2 ON r.target_id = c2.id
        """)
        self.relations_by_type = {}
        for src, tgt, rtype, weight in cur.fetchall():
            self.relations_by_type.setdefault(rtype, []).append((src, tgt, weight))

        self.all_entities = self.entities_by_domain.get(1, []) + self.entities_by_domain.get(2, []) + self.entities_by_domain.get(3, [])
        self.locations = [tgt for src, tgt, w in self.relations_by_type.get(6, [])]
        self.capable_of_pairs = self.relations_by_type.get(5, [])
        self.has_property_pairs = self.relations_by_type.get(4, [])

    def _gen_relation_prompts(self, relation_type_id):
        templates = _RELATION_TEMPLATES.get(relation_type_id, [])
        pairs = self.relations_by_type.get(relation_type_id, [])
        if not templates or not pairs:
            return []
        prompts = []
        for src, tgt, w in pairs:
            for t in templates:
                prompts.append(t.format(source=src, target=tgt))
        return prompts

    def _gen_property_category_prompts(self):
        prompts = []
        for src, tgt, w in self.has_property_pairs:
            for cid, props in self.properties_by_category.items():
                if tgt in props:
                    templates = _PROPERTY_QUERY_TEMPLATES.get(cid, [])
                    for t in templates:
                        prompts.append(t.format(source=src))
                    break
        return prompts

    def _gen_scene_prompts(self, max_prompts=5000):
        prompts = set()
        adj_map = {}
        for src, tgt, w in self.has_property_pairs:
            adj_map.setdefault(src, []).append(tgt)
        entities_with_adj = [e for e in self.all_entities if e in adj_map]
        locs = self.locations if self.locations else ["a room"]
        # Sample single-entity scenes instead of full cross-product
        self.rng.shuffle(entities_with_adj)
        for ent in entities_with_adj:
            if len(prompts) >= max_prompts:
                break
            adj = self.rng.choice(adj_map[ent])
            loc = self.rng.choice(locs)
            t = self.rng.choice(_SCENE_TEMPLATES_SINGLE)
            prompts.add(t.format(adj=adj, entity=ent, location=loc))
        # Sample two-entity scenes
        if len(entities_with_adj) >= 2:
            attempts = min(max_prompts - len(prompts), len(entities_with_adj) * 2)
            for _ in range(max(0, attempts)):
                e1, e2 = self.rng.sample(entities_with_adj, 2)
                a1 = self.rng.choice(adj_map[e1])
                a2 = self.rng.choice(adj_map[e2])
                loc = self.rng.choice(locs)
                t = self.rng.choice(_SCENE_TEMPLATES_DOUBLE)
                prompts.add(t.format(entity1=e1, entity2=e2, adj1=a1, adj2=a2, location=loc))
        return list(prompts)

    def _gen_negation_prompts(self, max_prompts=5000):
        prompts = set()
        capable_entities = list(set(src for src, tgt, w in self.capable_of_pairs))
        capable_actions = list(set(tgt for src, tgt, w in self.capable_of_pairs))
        prop_entities = list(set(src for src, tgt, w in self.has_property_pairs))
        prop_values = list(set(tgt for src, tgt, w in self.has_property_pairs))
        entity_actions = {}
        for src, tgt, w in self.capable_of_pairs:
            entity_actions.setdefault(src, set()).add(tgt)
        entity_props = {}
        for src, tgt, w in self.has_property_pairs:
            entity_props.setdefault(src, set()).add(tgt)
        neg_templates_cap = [t for t in _NEGATION_TEMPLATES if "{action}" in t]
        neg_templates_prop = [t for t in _NEGATION_TEMPLATES if "{property}" in t]
        # Sample CapableOf negations instead of full cross-product
        if capable_entities and capable_actions and neg_templates_cap:
            for _ in range(max_prompts * 2):
                if len(prompts) >= max_prompts // 2:
                    break
                ent = self.rng.choice(capable_entities)
                act = self.rng.choice(capable_actions)
                if act not in entity_actions.get(ent, set()):
                    t = self.rng.choice(neg_templates_cap)
                    prompts.add(t.format(entity=ent, action=act))
        # Sample HasProperty negations
        if prop_entities and prop_values and neg_templates_prop:
            for _ in range(max_prompts * 2):
                if len(prompts) >= max_prompts:
                    break
                ent = self.rng.choice(prop_entities)
                prop = self.rng.choice(prop_values)
                if prop not in entity_props.get(ent, set()):
                    t = self.rng.choice(neg_templates_prop)
                    prompts.add(t.format(entity=ent, property=prop))
        return list(prompts)

    def _gen_comparison_prompts(self, max_prompts=5000):
        prompts = set()
        by_cat = {}
        for cid, info in self.concepts.items():
            if info["domain"] in (1, 2, 3):
                by_cat.setdefault(info["category"], []).append(info["surface_text"])
        valid_groups = [ents for ents in by_cat.values() if len(ents) >= 2]
        if not valid_groups:
            return []
        for _ in range(max_prompts * 2):
            if len(prompts) >= max_prompts:
                break
            group = self.rng.choice(valid_groups)
            e1, e2 = self.rng.sample(group, 2)
            t = self.rng.choice(_COMPARISON_TEMPLATES)
            prompts.add(t.format(entity1=e1, entity2=e2))
        return list(prompts)

    def _gen_entity_prompts(self, max_prompts=5000):
        prompts = set()
        entities = list(self.all_entities)
        self.rng.shuffle(entities)
        for ent in entities:
            if len(prompts) >= max_prompts:
                break
            t = self.rng.choice(_ENTITY_TEMPLATES)
            prompts.add(t.format(entity=ent))
        return list(prompts)

    def generate(self, num_prompts):
        micro_count = len(MICRO_PROMPTS)
        base = list(MICRO_PROMPTS)

        if num_prompts <= micro_count:
            return base[:num_prompts]

        remaining = num_prompts - micro_count
        pool_cap = max(remaining * 3, 15000)

        pools = {
            "has_property_direct": self._gen_relation_prompts(4),
            "has_property_category": self._gen_property_category_prompts(),
            "capable_of": self._gen_relation_prompts(5),
            "at_location": self._gen_relation_prompts(6),
            "is_a": self._gen_relation_prompts(1),
            "causes": self._gen_relation_prompts(7),
            "used_for": self._gen_relation_prompts(12),
            "scene": self._gen_scene_prompts(max_prompts=pool_cap),
            "negation": self._gen_negation_prompts(max_prompts=pool_cap),
            "comparison": self._gen_comparison_prompts(max_prompts=pool_cap),
            "minor_relations": (
                self._gen_relation_prompts(16)
                + self._gen_relation_prompts(3)
                + self._gen_relation_prompts(18)
                + self._gen_relation_prompts(22)
            ),
            "entity_only": self._gen_entity_prompts(max_prompts=pool_cap),
        }

        for key in pools:
            seen = set()
            deduped = []
            for p in pools[key]:
                low = p.lower()
                if low not in seen:
                    seen.add(low)
                    deduped.append(p)
            pools[key] = deduped
            self.rng.shuffle(pools[key])

        generated = []
        base_lower = set(p.lower() for p in base)
        overflow_pool = []

        for cat, pct in _TARGET_DISTRIBUTION.items():
            target = int(remaining * pct)
            pool = pools.get(cat, [])
            picked = [p for p in pool[:target] if p.lower() not in base_lower]
            generated.extend(picked)
            if len(pool) > target:
                overflow_pool.extend(pool[target:])

        self.rng.shuffle(overflow_pool)
        seen_all = base_lower | set(p.lower() for p in generated)
        for p in overflow_pool:
            if len(generated) + micro_count >= num_prompts:
                break
            if p.lower() not in seen_all:
                generated.append(p)
                seen_all.add(p.lower())

        shortfall = num_prompts - micro_count - len(generated)
        if shortfall > 0:
            all_gen = list(generated)
            idx = 0
            while shortfall > 0:
                generated.append(all_gen[idx % len(all_gen)])
                idx += 1
                shortfall -= 1

        self.rng.shuffle(generated)
        return base + generated

    def close(self):
        self.conn.close()

print(f"PromptGenerator defined ({len(_RELATION_TEMPLATES)} relation types, {len(_PROPERTY_QUERY_TEMPLATES)} property categories)")

In [ ]:
def _parse_teacher_response(response):
    thinking = answer = ''
    if '<think>' in response and '</think>' in response:
        thinking = response[response.index('<think>')+10:response.index('</think>')].strip()
    if '<response>' in response and '</response>' in response:
        answer = response[response.index('<response>')+10:response.index('</response>')].strip()
    if not thinking and not answer:
        lines = response.strip().split('\n')
        if len(lines) > 2:
            thinking = '\n'.join(lines[:len(lines)//2])
            answer = '\n'.join(lines[len(lines)//2:])
        else:
            return '', ''
    return thinking, answer


_PUNT_PHRASES = (
    'not enough information',
    'does not contain sufficient',
    'does not contain information',
    'does not provide sufficient',
    'does not provide enough',
    'does not provide information',
    'does not provide a direct',
    'does not directly state',
    'does not directly provide',
    'not possible to determine',
    'cannot be determined',
    'cannot determine',
    "doesn't directly state",
    'no information available',
    'insufficient information',
)


def _is_punt_response(response):
    lower = response.lower()
    return any(phrase in lower for phrase in _PUNT_PHRASES)


class _RateLimiter:
    """Token-bucket rate limiter for Groq API calls."""
    def __init__(self, rpm_target, max_concurrent):
        self._interval = 60.0 / rpm_target
        self._sem = asyncio.Semaphore(max_concurrent)
        self._last = 0.0

    async def acquire(self):
        await self._sem.acquire()
        now = time.monotonic()
        wait = self._last + self._interval - now
        if wait > 0:
            await asyncio.sleep(wait)
        self._last = time.monotonic()

    def release(self):
        self._sem.release()


async def _call_groq_with_retry(client, messages, config, max_retries=5, initial_backoff=1.0):
    """Call Groq API with exponential backoff retry."""
    backoff = initial_backoff
    for attempt in range(max_retries):
        try:
            completion = client.chat.completions.create(
                model=config['model'],
                messages=messages,
                max_tokens=config['max_tokens'],
                temperature=config['temperature'],
            )
            return completion.choices[0].message.content
        except Exception as e:
            err_str = str(e)
            if 'rate_limit' in err_str.lower() or '429' in err_str:
                wait = backoff * (2 ** attempt) + random.uniform(0, 1)
                await asyncio.sleep(wait)
            else:
                if attempt == max_retries - 1:
                    raise
                await asyncio.sleep(backoff)
    return None


async def _process_one(idx, prompt, encoder, client, groq_system_msg, limiter):
    """Process a single prompt: encode → call Groq → parse → return example."""
    try:
        sml_block = encoder.encode(prompt)
        teacher_prompt = TEACHER_PROMPT_TEMPLATE.format(prompt=prompt, sml_block=sml_block)

        await limiter.acquire()
        try:
            teacher_response = await asyncio.get_event_loop().run_in_executor(
                None,
                lambda: _call_groq_with_retry.__wrapped__(client,
                    [{'role': 'system', 'content': groq_system_msg},
                     {'role': 'user', 'content': teacher_prompt}],
                    GROQ_CONFIG)
            )
        finally:
            limiter.release()

        if not teacher_response:
            return None

        thinking, response = _parse_teacher_response(teacher_response)
        if not thinking or not response:
            return None

        assistant_content = f'{sml_block}\n<think>\n{thinking}\n</think>\n<response>\n{response}\n</response>'
        return {'messages': [
            {'role': 'system', 'content': SML_SYSTEM_PROMPT},
            {'role': 'user', 'content': prompt},
            {'role': 'assistant', 'content': assistant_content},
        ]}
    except Exception as e:
        return None


# Synchronous wrapper that uses Groq sync client with rate limiting
def generate_training_data(num_examples=1500):
    """Generate training data via Groq teacher model with rate limiting."""
    from groq import Groq
    from tqdm.notebook import tqdm

    encoder = SMLEncoder(str(BIBLE_DB_PATH))
    client = Groq(api_key=GROQ_API_KEY)

    if num_examples <= len(MICRO_PROMPTS):
        prompts = MICRO_PROMPTS[:num_examples]
    else:
        gen = PromptGenerator(str(BIBLE_DB_PATH))
        prompts = gen.generate(num_examples)
        gen.close()
        unique_count = len(set(p.lower() for p in prompts))
        print(f'  Using PromptGenerator: {len(prompts)} prompts ({unique_count} unique)')

    groq_system_msg = (
        'You are a neurosymbolic reasoning assistant. You will be given a user question '
        'and a Semantic Markup Language (SML) context block that contains grounded facts.\n\n'
        'You must respond in EXACTLY this format:\n\n'
        '<think>\n'
        'SML entities identified: [list the entities from the SML block with their anchor tokens]\n'
        'SML relations: [list the relations and what they mean]\n'
        'Reasoning: [explain your reasoning, explicitly referencing the SML data]\n'
        '</think>\n'
        '<response>\n'
        '[Your answer to the user\'s question, grounded in the SML facts]\n'
        '</response>\n\n'
        'CRITICAL RULES:\n'
        '- Your response MUST be grounded in the SML context provided\n'
        '- You MUST reference specific SML anchor tokens (e.g., dog_1001, yellow_3004) in your thinking\n'
        '- If the SML says something that contradicts common knowledge, FOLLOW THE SML\n'
        '- The <think> block must be at least 2-3 sentences\n'
        '- Never skip the <think> block\n'
        '- If a relation uses NOT_ prefix (e.g., NOT_CapableOf), it means the entity CANNOT do that action'
    )

    # Rate limiter
    rpm = GROQ_PARALLEL['rpm_target']
    interval = 60.0 / rpm
    generated, failed = 0, 0
    unknown_budget = max(1, int(num_examples * 0.05))
    thin_budget = max(5, int(num_examples * 0.20))
    unknown_count = 0
    thin_count = 0
    output_path = str(TRAINING_DATA_PATH)

    with open(output_path, 'w') as f:
        for prompt in tqdm(prompts, desc='Generating'):
            try:
                sml_block = encoder.encode(prompt)

                # Quality gate
                block_lines = sml_block.strip().split('\n')
                has_relation = any(l.strip().startswith('R(') for l in block_lines)
                has_known = any('E(' in l and 'unknown_' not in l for l in block_lines)

                if has_known and has_relation:
                    pass  # rich — always keep
                elif has_known and thin_count < thin_budget:
                    thin_count += 1  # thin — keep within budget
                elif not has_known and unknown_count < unknown_budget:
                    unknown_count += 1  # unknown — keep as fallback example
                else:
                    failed += 1
                    continue

                teacher_prompt = TEACHER_PROMPT_TEMPLATE.format(prompt=prompt, sml_block=sml_block)

                completion = client.chat.completions.create(
                    model=GROQ_CONFIG['model'],
                    messages=[
                        {'role': 'system', 'content': groq_system_msg},
                        {'role': 'user', 'content': teacher_prompt},
                    ],
                    max_tokens=GROQ_CONFIG['max_tokens'],
                    temperature=GROQ_CONFIG['temperature'],
                )

                teacher_response = completion.choices[0].message.content
                if not teacher_response: failed += 1; continue

                thinking, response = _parse_teacher_response(teacher_response)
                if not thinking or not response: failed += 1; continue
                if _is_punt_response(response): failed += 1; continue

                assistant_content = f'{sml_block}\n<think>\n{thinking}\n</think>\n<response>\n{response}\n</response>'
                example = {'messages': [
                    {'role': 'system', 'content': SML_SYSTEM_PROMPT},
                    {'role': 'user', 'content': prompt},
                    {'role': 'assistant', 'content': assistant_content},
                ]}
                f.write(json.dumps(example) + '\n')
                generated += 1

            except Exception as e:
                err_str = str(e)
                if 'rate_limit' in err_str.lower() or '429' in err_str:
                    time.sleep(5.0)  # Back off on rate limit
                    failed += 1
                else:
                    print(f'\nError: {e}')
                    failed += 1

            time.sleep(interval)  # Rate limit pacing

    encoder.close()
    print(f'\nDone: {generated} generated, {failed} failed')
    return output_path


In [ ]:
# ── Generate! ─────────────────────────────────────────────────────
# Uses NUM_EXAMPLES from run mode config above.

print(f'Generating {NUM_EXAMPLES} examples...')
output_path = generate_training_data(num_examples=NUM_EXAMPLES)

In [ ]:
# Quick check: how many examples and a sample
with open(str(TRAINING_DATA_PATH)) as f:
    lines = f.readlines()
print(f'Total training examples: {len(lines)}')

sample = json.loads(lines[0])
print(f'\nSample assistant content (first 500 chars):')
print(sample['messages'][2]['content'][:500])

### 6b. (Optional) Upload pre-generated data

If you already have `training_data.jsonl`, upload it instead of running generation:

In [ ]:
# # Uncomment to upload your own JSONL:
# from google.colab import files
# uploaded = files.upload()  # Select your training_data.jsonl
# import shutil
# for name in uploaded:
#     shutil.move(name, str(TRAINING_DATA_PATH))
#     print(f'Uploaded {name} → {TRAINING_DATA_PATH}')

## 7. Train with QLoRA + Unsloth

In [ ]:
from unsloth import FastLanguageModel
from datasets import Dataset

# ── Load model ────────────────────────────────────────────────────
print('Loading model...')
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=TRAINING_CONFIG['model_name'],
    max_seq_length=TRAINING_CONFIG['max_seq_length'],
    load_in_4bit=True,
)

# ── Configure LoRA ─────────────────────────────────────────────────
print('Configuring LoRA...')
model = FastLanguageModel.get_peft_model(
    model,
    r=TRAINING_CONFIG['lora_r'],
    lora_alpha=TRAINING_CONFIG['lora_alpha'],
    target_modules=TRAINING_CONFIG['target_modules'],
    lora_dropout=TRAINING_CONFIG['lora_dropout'],
    bias='none',
    use_gradient_checkpointing='unsloth',
)

print(f'Trainable params: {model.print_trainable_parameters()}')

In [ ]:
# ── Load dataset ──────────────────────────────────────────────────
examples = []
with open(str(TRAINING_DATA_PATH)) as f:
    for line in f:
        line = line.strip()
        if line:
            examples.append(json.loads(line))

records = [{'messages': ex['messages']} for ex in examples]
dataset = Dataset.from_list(records)
split = dataset.train_test_split(test_size=0.1, seed=42)

print(f'Train: {len(split["train"])}, Test: {len(split["test"])}')

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

output_dir = str(MODEL_OUTPUT_DIR)
Path(output_dir).mkdir(parents=True, exist_ok=True)

training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=TRAINING_CONFIG['per_device_train_batch_size'],
    gradient_accumulation_steps=TRAINING_CONFIG['gradient_accumulation_steps'],
    learning_rate=TRAINING_CONFIG['learning_rate'],
    lr_scheduler_type=TRAINING_CONFIG['lr_scheduler_type'],
    warmup_ratio=TRAINING_CONFIG['warmup_ratio'],
    weight_decay=TRAINING_CONFIG['weight_decay'],
    bf16=True,
    logging_steps=10,
    save_strategy='steps',
    save_steps=500,
    eval_strategy='steps',
    eval_steps=500,
    save_total_limit=3,
    report_to='none',
)

def formatting_func(examples):
    messages = examples['messages']
    if messages and isinstance(messages[0], dict):
        return [tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)]
    return [tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False) for msgs in messages]

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=split['train'],
    eval_dataset=split['test'],
    args=training_args,
    formatting_func=formatting_func,
    max_seq_length=TRAINING_CONFIG['max_seq_length'],
)

print(f'Trainer ready. Starting training ({NUM_EPOCHS} epochs)...')

In [ ]:
# ── Train! ────────────────────────────────────────────────────────
trainer.train()

In [ ]:
# ── Save LoRA adapter + merged model ──────────────────────────────
adapter_path = str(MODEL_OUTPUT_DIR / 'sml_adapter')
model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
print(f'LoRA adapter saved to {adapter_path}')

merged_path = str(MODEL_OUTPUT_DIR / 'sml_merged')
model.save_pretrained_merged(merged_path, tokenizer)
print(f'Merged model saved to {merged_path}')

## 8. Inference Test

In [ ]:
def run_inference(user_input, model, tokenizer, encoder, custom_sml=None):
    """Run the full SML inference pipeline."""
    sml_block = custom_sml if custom_sml else encoder.encode(user_input)

    messages = [
        {'role': 'system', 'content': SML_SYSTEM_PROMPT},
        {'role': 'user', 'content': user_input},
    ]
    prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    prompt_text += sml_block + '\n<think>\n'  # Force thinking block

    inputs = tokenizer(prompt_text, return_tensors='pt').to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=1024, do_sample=True,
            temperature=0.7, top_p=0.9, pad_token_id=tokenizer.eos_token_id)

    new_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    raw = tokenizer.decode(new_tokens, skip_special_tokens=True)

    # Parse
    thinking = response = ''
    m = re.search(r'<think>(.*?)</think>', sml_block + '\n' + raw, re.DOTALL)
    if m: thinking = m.group(1).strip()
    # Also check if raw starts inside a thinking block (since we forced the opening tag)
    if not thinking:
        m = re.search(r'(.*?)</think>', raw, re.DOTALL)
        if m: thinking = m.group(1).strip()
    m = re.search(r'<response>(.*?)</response>', raw, re.DOTALL)
    if m: response = m.group(1).strip()
    if not response:
        response = re.sub(r'<.*?>', '', raw).strip()

    return {'sml_block': sml_block, 'thinking': thinking, 'response': response, 'raw': raw}


# Load merged model for inference
FastLanguageModel.for_inference(model)

encoder = SMLEncoder(str(BIBLE_DB_PATH))

# ── Test queries ──────────────────────────────────────────────────
test_queries = [
    'What color is the sun?',
    'Can birds fly?',
    'Where do fish live?',
    'Is an elephant big?',
]

for q in test_queries:
    result = run_inference(q, model, tokenizer, encoder)
    print(f'Q: {q}')
    print(f'  SML: {result["sml_block"][:80]}...')
    print(f'  Thinking: {result["thinking"][:100]}...')
    print(f'  Response: {result["response"][:100]}')
    print()

## 9. Liar Ablation Test

The critical test: inject **false SML** and verify the model follows the lie.

In [ ]:
liar_tests = [
    ('What color is the sun?', 'green',
     '<sml>\nE(1|5|0|0|sun_8001|green_3003|0|0.95)\nR(HasProperty|0|0|0.95|0|0)\n</sml>'),
    ('What color is the sky?', 'red',
     '<sml>\nE(1|5|0|0|sky_8002|red_3001|0|0.95)\nR(HasProperty|0|0|0.95|0|0)\n</sml>'),
    ('Can birds fly?', 'no',
     '<sml>\nE(1|1|2|2|bird_1003|0|0|0.90)\nE(3|0|0|0|fly_5004|0|0|0.90)\nR(NOT_CapableOf|0|1|0.90|0|0)\n</sml>'),
    ('Is an elephant big?', 'small',
     '<sml>\nE(1|1|2|3|elephant_1007|small_3102|0|0.95)\nR(HasProperty|0|0|0.95|0|0)\n</sml>'),
    ('Where do fish live?', 'park',
     '<sml>\nE(1|1|2|4|fish_1004|0|0|0.90)\nE(1|4|0|0|park_4001|0|0|0.90)\nR(AtLocation|0|1|0.95|0|0)\n</sml>'),
]

print('LIAR ABLATION TEST')
print('='*60)
passed = 0
for prompt, expected, false_sml in liar_tests:
    result = run_inference(prompt, model, tokenizer, encoder, custom_sml=false_sml)
    combined = (result['response'] + ' ' + result['thinking']).lower()
    ok = expected.lower() in combined
    passed += ok
    print(f'  [{"PASS" if ok else "FAIL"}] {prompt} → expected "{expected}"')
    print(f'         Response: {result["response"][:100]}')

print(f'\nLiar Ablation: {passed}/{len(liar_tests)} ({100*passed//len(liar_tests)}%)')
if passed >= len(liar_tests) // 2:
    print('>>> GROUNDING IS WORKING: Model follows SML context over pre-trained weights.')
else:
    print('>>> GROUNDING NEEDS WORK: Model is ignoring SML.')

## 10. Save & Upload to HuggingFace

In [ ]:
# ── Save locally + Push merged model to HuggingFace ───────────────
# Authenticate with HuggingFace
from huggingface_hub import login
from google.colab import userdata

try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
    print('Authenticated with HuggingFace')
except Exception as e:
    print(f'HF auth failed: {e}')
    print('Set HF_TOKEN in Colab Secrets (key icon in left sidebar)')
    hf_token = None

# Save LoRA adapter locally
adapter_path = str(MODEL_OUTPUT_DIR / 'sml_adapter')
model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
print(f'LoRA adapter saved to {adapter_path}')

# Save merged model locally
merged_path = str(MODEL_OUTPUT_DIR / 'sml_merged')
model.save_pretrained_merged(merged_path, tokenizer)
print(f'Merged model saved to {merged_path}')

# Push MERGED model to HuggingFace Hub
if hf_token:
    print(f'\nPushing merged model to {HF_REPO}...')
    model.push_to_hub_merged(
        HF_REPO,
        tokenizer,
        save_method='merged_16bit',
        token=hf_token,
    )
    print(f'Merged model pushed to https://huggingface.co/{HF_REPO}')
else:
    print('\nSkipping HuggingFace push (no token). Set HF_TOKEN in Colab Secrets.')

In [ ]:
encoder.close()
print('Done!')